# LLM DL Ensembles

Ноутбук строит ансамбли для DL-моделей с LLM-признаками.
Входные таблицы и артефакты задаются через переменные окружения, результаты сохраняются в отдельную папку запуска.

In [ ]:
import os

TABPFN_ENABLE_HF_LOGIN = False
TABPFN_FORCE_MODEL_VERSION = "V2_5"
TABPFN_HF_TOKEN_EFFECTIVE = os.environ.get("HF_TOKEN", "").strip()
TABPFN_HF_LOGIN_STATUS = "not_used"


In [ ]:

import warnings
warnings.filterwarnings("ignore")

import os
import re
import ast
import json
import time
import math
import gc
import random
from datetime import datetime
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

from scipy.optimize import nnls, minimize

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import MiniBatchKMeans
from sklearn.mixture import GaussianMixture
from sklearn.metrics import mean_absolute_error, mean_squared_error, median_absolute_error
from sklearn.metrics import pairwise_distances

from sklearn.linear_model import (
    Ridge,
    Lasso,
    ElasticNet,
    BayesianRidge,
    HuberRegressor,
    LinearRegression,
    QuantileRegressor,
)
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor

try:
    from xgboost import XGBRegressor
    HAS_XGB_FOR_STACKING = True
except Exception:
    HAS_XGB_FOR_STACKING = False
    XGBRegressor = None

seed = 2
np.random.seed(seed)


In [ ]:

target_col = "duration_hours"
cap = 960
SEED = 2
seed = SEED
np.random.seed(SEED)

REMOTE_RUNTIME = False
USE_REMOTE_STORAGE = False
MOUNT_REMOTE_STORAGE = False
DRIVE_ROOT = Path(".")
DRIVE_PROJECT_DIR = Path(".")

DATA_PATH_ENV = "DATA_FINALL_PATH"
ARTIFACT_ROOT_ENV = "LLM_PROVIDER_FUSION_DL_ARTIFACTS_DIR"
PROVIDER_ENV_ROOT_VARS = {
    "gpt": "GPT_DL_ARTIFACT_ROOT",
    "claude": "CLAUDE_DL_ARTIFACT_ROOT",
    "ollama": "OLLAMA_DL_ARTIFACT_ROOT",
    "qwen": "QWEN_DL_ARTIFACT_ROOT",
}


def require_path(env_name: str, label: str) -> Path:
    value = os.environ.get(env_name)
    if not value:
        raise RuntimeError(f"Укажи путь к {label} в переменной окружения {env_name}.")
    path = Path(value).expanduser()
    if not path.exists():
        raise FileNotFoundError(f"Файл для {label} не найден: {path}")
    return path


DATA_PATH = require_path(DATA_PATH_ENV, "data_finall")
ARTIFACT_ROOT = Path(os.environ.get(ARTIFACT_ROOT_ENV, "./artifacts_llm_provider_fusion_dl_ensembles_final")).expanduser()
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

RUN_NAME = datetime.now().strftime("run_%Y%m%d_%H%M%S")
RUN_DIR = ARTIFACT_ROOT / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)

PROVIDER_MANUAL_ARTIFACT_ROOTS = {provider: os.environ.get(env_name, "") for provider, env_name in PROVIDER_ENV_ROOT_VARS.items()}

KNOWN_ARTIFACT_ROOT_NAMES = {
    "gpt": [
        "artifacts_gpt_api",
        "artifacts",
        "artifacts_gpt",
        "artifacts_gpt_pristine",
    ],
    "claude": [
        "artifacts_claude_api",
    ],
    "ollama": [
        "artifacts_ollama_local_pristine",
        "artifacts_ollama_local",
    ],
    "qwen": [
        "artifacts_qwen_local_pristine",
        "artifacts_qwen_local",
    ],
}

SEARCH_ROOTS = [Path(".")]

FEATURE_FILE_PATTERNS = {
    "gpt": [
        "llm_features_{variant}_{split}.csv",
        "llm_features_gpt*_{variant}_{split}.csv",
    ],
    "claude": [
        "llm_features_claude_api*_{variant}_{split}.csv",
    ],
    "ollama": [
        "llm_features_ollama*_{variant}_{split}.csv",
    ],
    "qwen": [
        "llm_features_qwen_local*_{variant}_{split}.csv",
    ],
}

SCHEME_FAMILY_ORDER = ["llm_only", "cluster_before_llm", "llm_then_cluster"]

FIXED_CLUSTER_CONFIGS = [
    {
        "tag": "gmm_diag_5",
        "clusterer": "GaussianMixture",
        "params": {"n_components": 5, "covariance_type": "diag"},
    },
    {
        "tag": "mbkm_2",
        "clusterer": "MiniBatchKMeans",
        "params": {"n_clusters": 2},
    },
]

MODEL_INCLUDE = None
KEEP_ONLY_POSITIVE_DELTA = False
MAX_BASE_LEARNERS = None

# Outer split for ensemble construction:
# first BLEND_TRAIN_FRAC of train_typical is used to train base models,
# the rest is used as blend-validation matrix.
BLEND_TRAIN_FRAC = 0.85
BLEND_FIT_FRAC = 0.50

# Inner DL validation inside each rebuilt base learner.
DL_INNER_VAL_FRAC = 0.20

RUN_ALL_PAIRS = True
RUN_ALL_TRIPLES = True
RUN_TOPK_PREFIX_ENSEMBLES = True
RUN_GREEDY_SEARCH = True
RUN_STACKERS = True
RUN_EXHAUSTIVE_TOPN_SUBSETS = True
EXHAUSTIVE_TOP_N = 10
REFIT_TOP_ENSEMBLES = 300
PAIR_WEIGHT_GRID = np.linspace(0.0, 1.0, 101)

FORCE_REFRESH_BASE_PREDICTIONS = False
RESUME_IF_PREDICTIONS_EXIST = True



def seed_everything(seed: int = 2):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    try:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed(seed)
            torch.cuda.manual_seed_all(seed)
        if hasattr(torch.backends, "cudnn"):
            torch.backends.cudnn.deterministic = True
            torch.backends.cudnn.benchmark = False
    except Exception:
        pass

seed_everything(SEED)


In [ ]:
# ------------------------------------------------------------
# Save helpers
# ------------------------------------------------------------
def save_df(df: pd.DataFrame, name: str):
    run_path = RUN_DIR / name
    latest_path = ARTIFACT_ROOT / name.replace(".csv", "_latest.csv")
    df.to_csv(run_path, index=False)
    df.to_csv(latest_path, index=False)
    return run_path, latest_path

def save_json(payload, name: str):
    run_path = RUN_DIR / name
    latest_path = ARTIFACT_ROOT / name.replace(".json", "_latest.json")
    run_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
    latest_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
    return run_path, latest_path

def sanitize_name(name: str) -> str:
    return "".join(ch if ch.isalnum() or ch in "._-" else "_" for ch in str(name))

def clip_pred(pred):
    return np.clip(np.asarray(pred, dtype=float), 0, None)

def mae_metric(y, p):
    return float(mean_absolute_error(y, clip_pred(p)))

def rmse_metric(y, p):
    return float(np.sqrt(mean_squared_error(y, clip_pred(p))))

def mdae_metric(y, p):
    return float(median_absolute_error(y, clip_pred(p)))

def score_predictions(y_true, y_pred):
    return {
        "MAE": mae_metric(y_true, y_pred),
        "RMSE": rmse_metric(y_true, y_pred),
        "MdAE": mdae_metric(y_true, y_pred),
    }

In [ ]:

# ------------------------------------------------------------
# Provider run discovery
# ------------------------------------------------------------
def _dedupe_paths(paths):
    seen = set()
    out = []
    for p in paths:
        p = Path(p)
        try:
            key = str(p.resolve())
        except Exception:
            key = str(p)
        if key not in seen:
            out.append(p)
            seen.add(key)
    return out

def _candidate_sort_key(summary_path: Path, provider: str):
    try:
        mtime = -summary_path.stat().st_mtime
    except Exception:
        mtime = 0.0

    parent = summary_path.parent
    root_name = parent.parent.name if parent.name == "dl_tuning" else parent.name
    expected = KNOWN_ARTIFACT_ROOT_NAMES.get(provider, [])
    exact_bonus = 0 if root_name in expected else 1
    return (exact_bonus, mtime, len(str(summary_path)), str(summary_path))

def _record_from_summary(summary_path: Path):
    summary_path = Path(summary_path)
    summary_dir = summary_path.parent
    if summary_dir.name == "dl_tuning":
        artifact_root = summary_dir.parent
        feature_dir = summary_dir.parent
    else:
        artifact_root = summary_dir
        feature_dir = summary_dir

    return {
        "summary_path": summary_path,
        "summary_dir": summary_dir,
        "artifact_root": artifact_root,
        "feature_dir": feature_dir,
    }

def _summary_candidates_from_root_like(raw):
    if raw is None:
        return []

    p = Path(raw)
    candidates = []

    if p.is_file():
        if p.name in {"dl_llm_cluster_compare_results.csv", "llm_cluster_compare_results.csv"}:
            candidates.append(p)
    elif p.is_dir():
        for rel in [
            Path("dl_tuning") / "dl_llm_cluster_compare_results.csv",
            Path("dl_tuning") / "llm_cluster_compare_results.csv",
            Path("dl_llm_cluster_compare_results.csv"),
            Path("llm_cluster_compare_results.csv"),
        ]:
            cand = p / rel
            if cand.exists():
                candidates.append(cand)

    return _dedupe_paths(candidates)

def find_provider_summary_candidates(provider: str):
    manual_candidates = []
    env_candidates = []
    auto_candidates = []

    manual = PROVIDER_MANUAL_ARTIFACT_ROOTS.get(provider)
    if manual:
        manual_candidates.extend(_summary_candidates_from_root_like(manual))

    env_var = PROVIDER_ENV_ROOT_VARS.get(provider)
    if env_var:
        env_value = os.getenv(env_var)
        if env_value:
            env_candidates.extend(_summary_candidates_from_root_like(env_value))

    if manual_candidates:
        return sorted(_dedupe_paths(manual_candidates), key=lambda p: _candidate_sort_key(p, provider))
    if env_candidates:
        return sorted(_dedupe_paths(env_candidates), key=lambda p: _candidate_sort_key(p, provider))

    for root in SEARCH_ROOTS:
        if not root.exists():
            continue
        for dir_name in KNOWN_ARTIFACT_ROOT_NAMES.get(provider, []):
            patterns = [
                f"**/{dir_name}/dl_tuning/dl_llm_cluster_compare_results.csv",
                f"**/{dir_name}/dl_tuning/llm_cluster_compare_results.csv",
                f"**/{dir_name}/dl_llm_cluster_compare_results.csv",
                f"**/{dir_name}/llm_cluster_compare_results.csv",
            ]
            for pattern in patterns:
                try:
                    auto_candidates.extend(root.glob(pattern))
                except Exception:
                    pass

    auto_candidates = [Path(p) for p in auto_candidates if Path(p).exists()]
    auto_candidates = _dedupe_paths(auto_candidates)
    auto_candidates = sorted(auto_candidates, key=lambda p: _candidate_sort_key(p, provider))
    return auto_candidates

def resolve_provider_registry():
    rows = []
    registry = {}
    for provider in ["gpt", "claude", "ollama", "qwen"]:
        cands = find_provider_summary_candidates(provider)
        record = _record_from_summary(cands[0]) if cands else None

        rows.append({
            "provider": provider,
            "status": "found" if record is not None else "missing",
            "artifact_root": str(record["artifact_root"]) if record is not None else "",
            "feature_dir": str(record["feature_dir"]) if record is not None else "",
            "summary_dir": str(record["summary_dir"]) if record is not None else "",
            "summary_path": str(record["summary_path"]) if record is not None else "",
            "n_candidates_found": len(cands),
            "all_candidates": json.dumps([str(p) for p in cands], ensure_ascii=False),
        })

        if record is not None:
            registry[provider] = {
                "summary_path": record["summary_path"],
                "summary_dir": record["summary_dir"],
                "artifact_root": record["artifact_root"],
                "feature_dir": record["feature_dir"],
                "all_candidates": cands,
            }

    reg_df = pd.DataFrame(rows)
    return registry, reg_df

PROVIDER_REGISTRY, provider_registry_df = resolve_provider_registry()
display(provider_registry_df)

if len(PROVIDER_REGISTRY) == 0:
    raise FileNotFoundError(
        "Не найден ни один provider artifact dir. Заполни PROVIDER_MANUAL_ARTIFACT_ROOTS или *_DL_ARTIFACT_ROOT env vars."
    )

print("Resolved providers:", list(PROVIDER_REGISTRY.keys()))
save_df(provider_registry_df, "provider_registry.csv")


In [ ]:

# ------------------------------------------------------------
# Load data and reproduce the canonical split
# ------------------------------------------------------------
df = pd.read_csv(DATA_PATH)

if target_col not in df.columns:
    raise ValueError(f"В датасете нет целевой колонки {target_col!r}")

split = int(len(df) * 0.8)
train_full = df.iloc[:split].copy()
test_full = df.iloc[split:].copy()
test_typical = test_full[test_full[target_col] < cap].copy()

train_filt = train_full[train_full[target_col] < cap].copy()

meta_Xtr0 = train_filt.drop(columns=[target_col], errors="ignore").reset_index(drop=True)
meta_ytr0 = train_filt[target_col].reset_index(drop=True)

meta_Xte0 = test_full.drop(columns=[target_col], errors="ignore").reset_index(drop=True)
meta_yte0 = test_full[target_col].reset_index(drop=True)

meta_Xte_typ0 = test_typical.drop(columns=[target_col], errors="ignore").reset_index(drop=True)
meta_yte_typ0 = test_typical[target_col].reset_index(drop=True)

# bool -> 0/1 for sklearn / torch
for _frame in [meta_Xtr0, meta_Xte0, meta_Xte_typ0]:
    bool_cols = _frame.select_dtypes(include=["bool"]).columns.tolist()
    if bool_cols:
        _frame[bool_cols] = _frame[bool_cols].astype("int8")

llm_train_raw = train_filt.reset_index(drop=True).copy()
llm_test_raw = test_full.reset_index(drop=True).copy()
test_typical_mask = (llm_test_raw[target_col] < cap).reset_index(drop=True)

# aliases expected by the borrowed DL helpers
Xtrain = meta_Xtr0.copy()
ytrain = meta_ytr0.copy()
Xtest = meta_Xte_typ0.copy()
ytest = meta_yte_typ0.copy()
ytest_typical = meta_yte_typ0.copy()
Xtest_full = meta_Xte0.copy()
ytest_full = meta_yte0.copy()

print("train_full   :", train_full.shape)
print("train_filt   :", train_filt.shape)
print("test_full    :", test_full.shape)
print("test_typical :", test_typical.shape)
print("meta_Xtr0    :", meta_Xtr0.shape)
print("meta_Xte0    :", meta_Xte0.shape)
print("meta_Xte_typ0:", meta_Xte_typ0.shape)


In [ ]:
# ------------------------------------------------------------
# Cluster helpers and raw-space cluster variants
# ------------------------------------------------------------
def make_clusterer(name, params):
    if name == "MiniBatchKMeans":
        return MiniBatchKMeans(
            n_clusters=params["n_clusters"],
            random_state=seed,
            n_init=20,
            batch_size=1024,
        )
    elif name == "GaussianMixture":
        return GaussianMixture(
            n_components=params["n_components"],
            covariance_type=params["covariance_type"],
            random_state=seed,
        )
    else:
        raise ValueError(name)

def probs_from_dist(all_d):
    inv = 1.0 / (all_d + 1e-6)
    return inv / inv.sum(axis=1, keepdims=True)

def fit_clusterer_and_assign(name, params, Xtr, Xte):
    est = make_clusterer(name, params)

    if name == "MiniBatchKMeans":
        est.fit(Xtr)
        tr_labels = est.predict(Xtr).astype(int)
        te_labels = est.predict(Xte).astype(int)

        if len(np.unique(tr_labels)) < 2:
            return None

        tr_d = pairwise_distances(Xtr, est.cluster_centers_)
        te_d = pairwise_distances(Xte, est.cluster_centers_)
        tr_proba = probs_from_dist(tr_d)
        te_proba = probs_from_dist(te_d)
        return tr_labels, te_labels, tr_proba, te_proba, "native"

    if name == "GaussianMixture":
        est.fit(Xtr)
        tr_proba = est.predict_proba(Xtr)
        te_proba = est.predict_proba(Xte)

        tr_labels = np.argmax(tr_proba, axis=1).astype(int)
        te_labels = np.argmax(te_proba, axis=1).astype(int)

        if len(np.unique(tr_labels)) < 2:
            return None

        return tr_labels, te_labels, tr_proba, te_proba, "native"

    return None

def build_cluster_meta_features(tr_labels, te_labels, tr_proba=None, te_proba=None):
    tr_feat = pd.DataFrame({"cluster_id": tr_labels})
    te_feat = pd.DataFrame({"cluster_id": te_labels})

    tr_sizes = pd.Series(tr_labels).value_counts().to_dict()
    tr_feat["cluster_size_train"] = pd.Series(tr_labels).map(tr_sizes).fillna(0)
    te_feat["cluster_size_train"] = pd.Series(te_labels).map(tr_sizes).fillna(0)

    tr_feat["is_noise"] = (tr_feat["cluster_id"] == -1).astype(int)
    te_feat["is_noise"] = (te_feat["cluster_id"] == -1).astype(int)

    tr_ohe = pd.get_dummies(tr_feat["cluster_id"], prefix="cluster")
    te_ohe = pd.get_dummies(te_feat["cluster_id"], prefix="cluster").reindex(columns=tr_ohe.columns, fill_value=0)

    tr_feat = pd.concat([tr_feat, tr_ohe], axis=1)
    te_feat = pd.concat([te_feat, te_ohe], axis=1)

    if tr_proba is not None and te_proba is not None:
        for k in range(tr_proba.shape[1]):
            tr_feat[f"cluster_proba_{k}"] = tr_proba[:, k]
            te_feat[f"cluster_proba_{k}"] = te_proba[:, k]

    return tr_feat, te_feat

raw_cluster_scaler = StandardScaler()
raw_Xtr_scaled = raw_cluster_scaler.fit_transform(meta_Xtr0)
raw_Xte_scaled = raw_cluster_scaler.transform(meta_Xte0)

raw_cluster_pca = PCA(n_components=min(30, meta_Xtr0.shape[1]), random_state=seed)
raw_Xtr_embed = raw_cluster_pca.fit_transform(raw_Xtr_scaled)
raw_Xte_embed = raw_cluster_pca.transform(raw_Xte_scaled)

raw_cluster_variants = {}
raw_cluster_summary_rows = []

for cfg in FIXED_CLUSTER_CONFIGS:
    result = fit_clusterer_and_assign(cfg["clusterer"], cfg["params"], raw_Xtr_embed, raw_Xte_embed)
    if result is None:
        raise RuntimeError(f"Raw cluster config error: {cfg}")

    tr_labels, te_labels, tr_proba, te_proba, fit_mode = result
    tr_feat, te_feat = build_cluster_meta_features(tr_labels, te_labels, tr_proba, te_proba)

    raw_cluster_variants[cfg["tag"]] = {
        "cfg": cfg,
        "fit_mode": fit_mode,
        "tr_labels": tr_labels,
        "te_labels": te_labels,
        "tr_proba": tr_proba,
        "te_proba": te_proba,
        "train_feat_raw": tr_feat.reset_index(drop=True),
        "test_feat_raw": te_feat.reset_index(drop=True),
    }

    raw_cluster_summary_rows.append({
        "tag": cfg["tag"],
        "clusterer": cfg["clusterer"],
        "params": json.dumps(cfg["params"], ensure_ascii=False),
        "fit_mode": fit_mode,
        "n_clusters_train": int(len(np.unique(tr_labels))),
        "train_rows": int(len(tr_labels)),
        "test_rows": int(len(te_labels)),
    })

raw_cluster_summary_df = pd.DataFrame(raw_cluster_summary_rows)
display(raw_cluster_summary_df)
save_df(raw_cluster_summary_df, "raw_cluster_summary.csv")

In [ ]:

# ------------------------------------------------------------
# Load provider-level comparison tables and apply the 12-candidate logic
# ------------------------------------------------------------
def experiment_to_scheme_family(experiment_name: str):
    if experiment_name == "llm_only":
        return "llm_only"
    if str(experiment_name).startswith("cluster_before_llm__"):
        return "cluster_before_llm"
    if str(experiment_name).startswith("llm_then_cluster__"):
        return "llm_then_cluster"
    return None

def experiment_to_cluster_tag(experiment_name: str):
    if experiment_name == "llm_only":
        return None
    if "__" in str(experiment_name):
        return str(experiment_name).split("__", 1)[1]
    return None

required_cols = [
    "experiment",
    "model",
    "cv_MAE_internal",
    "holdout_typical_MAE",
    "holdout_full_MAE",
    "best_params",
]

provider_frames = []
for provider, info in PROVIDER_REGISTRY.items():
    part = pd.read_csv(info["summary_path"])

    # defensive compatibility
    if "cv_MAE_internal" not in part.columns and "cv_MAE" in part.columns:
        part = part.rename(columns={"cv_MAE": "cv_MAE_internal"})
    if "best_params" not in part.columns and "params" in part.columns:
        part = part.rename(columns={"params": "best_params"})
    if "epochs_used" not in part.columns and "epochs" in part.columns:
        part["epochs_used"] = part["epochs"]

    missing = [c for c in required_cols if c not in part.columns]
    if missing:
        raise ValueError(f"{provider}: missing columns in {info['summary_path']}: {missing}")

    part["provider"] = provider
    part["summary_path"] = str(info["summary_path"])
    part["artifact_root"] = str(info["artifact_root"])
    part["feature_dir"] = str(info["feature_dir"])
    part["scheme_family"] = part["experiment"].map(experiment_to_scheme_family)
    part["cluster_tag"] = part["experiment"].map(experiment_to_cluster_tag)

    if MODEL_INCLUDE is not None:
        part = part[part["model"].isin(MODEL_INCLUDE)].copy()

    provider_frames.append(part)

all_provider_rows = pd.concat(provider_frames, ignore_index=True)
all_provider_rows = all_provider_rows[all_provider_rows["scheme_family"].notna()].copy()

sort_cols = ["cv_MAE_internal", "holdout_typical_MAE", "holdout_full_MAE", "provider", "scheme_family", "experiment", "model"]
all_provider_rows = all_provider_rows.sort_values(sort_cols, kind="stable").reset_index(drop=True)

provider_family_model_best_df = (
    all_provider_rows
    .groupby(["provider", "scheme_family", "model"], as_index=False)
    .head(1)
    .reset_index(drop=True)
)

selected_best12_per_model_df = (
    provider_family_model_best_df
    .sort_values(sort_cols, kind="stable")
    .groupby("model", as_index=False)
    .head(1)
    .reset_index(drop=True)
)

provider_family_model_best_df["provider_family_id"] = (
    provider_family_model_best_df["provider"] + "::" + provider_family_model_best_df["scheme_family"]
)
selected_best12_per_model_df["provider_family_id"] = (
    selected_best12_per_model_df["provider"] + "::" + selected_best12_per_model_df["scheme_family"]
)
selected_best12_per_model_df["base_id"] = (
    selected_best12_per_model_df["provider"]
    + "::"
    + selected_best12_per_model_df["scheme_family"]
    + "::"
    + selected_best12_per_model_df["experiment"]
    + "::"
    + selected_best12_per_model_df["model"]
)

print("All provider rows:", len(all_provider_rows))
print("Best row per (provider, family, model):", len(provider_family_model_best_df))
print("Final selected base learners (best of up to 12 per architecture):", len(selected_best12_per_model_df))

display(
    provider_family_model_best_df[
        ["provider", "scheme_family", "experiment", "model", "cv_MAE_internal", "holdout_typical_MAE", "holdout_full_MAE"]
    ]
    .sort_values(["model", "cv_MAE_internal", "provider", "scheme_family"], kind="stable")
    .reset_index(drop=True)
)

display(
    selected_best12_per_model_df[
        ["base_id", "model", "provider", "scheme_family", "experiment", "cv_MAE_internal", "holdout_typical_MAE", "holdout_full_MAE"]
    ]
    .sort_values(["cv_MAE_internal", "holdout_typical_MAE", "holdout_full_MAE"], kind="stable")
    .reset_index(drop=True)
)

save_df(all_provider_rows, "provider_compare_all_rows.csv")
save_df(provider_family_model_best_df, "provider_family_model_best.csv")
save_df(selected_best12_per_model_df, "selected_best12_per_model.csv")

selection_payload = {
    "selection_level_1": "within each (provider, scheme_family, model) choose the best low-level experiment by cv_MAE_internal",
    "selection_level_2": "for each DL architecture choose the best candidate across provider x scheme_family (up to 12 candidates)",
    "providers_found": list(PROVIDER_REGISTRY.keys()),
    "n_all_rows": int(len(all_provider_rows)),
    "n_provider_family_model_best_rows": int(len(provider_family_model_best_df)),
    "n_final_base_learners": int(len(selected_best12_per_model_df)),
    "final_base_learners": selected_best12_per_model_df["base_id"].tolist(),
}
save_json(selection_payload, "selection_logic_summary.json")


## Rebuild exact feature spaces for the selected provider/scheme winners

Ниже notebook не дергает LLM API заново.
Он использует уже сохраненные `llm_features_*.csv` из provider-run'ов и заново конструирует только нужные feature spaces для финального reduced pool.


In [ ]:

# ------------------------------------------------------------
# Feature-file resolver and provider bundle reconstruction
# ------------------------------------------------------------
PROVIDER_BUNDLE_CACHE = {}
PROVIDER_FEATURE_PATH_LOG = {}

def normalize_llm_feature_frame(df_part: pd.DataFrame):
    if "row_id" not in df_part.columns:
        raise ValueError("feature csv must contain row_id")

    out = df_part.sort_values("row_id").reset_index(drop=True).copy()
    out = out.set_index("row_id")

    if "parse_error" not in out.columns:
        out["parse_error"] = 0

    numeric_cols = [c for c in out.columns if c != "parse_error"]
    for c in numeric_cols:
        out[c] = pd.to_numeric(out[c], errors="coerce")

    medians = out[numeric_cols].median(numeric_only=True)
    out[numeric_cols] = out[numeric_cols].fillna(medians)
    out["parse_error"] = out["parse_error"].fillna(0).astype(int)
    return out

def _feature_search_dirs(provider: str):
    info = PROVIDER_REGISTRY[provider]
    dirs = [
        info["feature_dir"],
        info["artifact_root"],
        info["summary_dir"],
        Path("./"),
                    ]
    if REMOTE_RUNTIME and USE_REMOTE_STORAGE:
        dirs.extend([
            DRIVE_ROOT,
                        DRIVE_PROJECT_DIR,
        ])
    return [p for p in _dedupe_paths(dirs) if Path(p).exists()]

def feature_file_candidates_for_provider(provider: str, variant_name: str, split_name: str):
    if provider not in PROVIDER_REGISTRY:
        raise KeyError(provider)

    patterns = [
        pat.format(variant=variant_name, split=split_name)
        for pat in FEATURE_FILE_PATTERNS[provider]
    ]

    matches = []
    for base in _feature_search_dirs(provider):
        for pat in patterns:
            try:
                matches.extend(sorted(base.glob(pat)))
            except Exception:
                pass

    matches = _dedupe_paths([p for p in matches if p.exists()])
    matches = sorted(
        matches,
        key=lambda p: (len(str(p.parent)), len(p.name), -p.stat().st_mtime if p.exists() else 0.0, str(p)),
    )
    return matches, patterns

def load_provider_variant_features(provider: str, variant_name: str, split_name: str):
    matches, patterns = feature_file_candidates_for_provider(provider, variant_name, split_name)
    if not matches:
        info = PROVIDER_REGISTRY[provider]
        sample = []
        for base in _feature_search_dirs(provider):
            try:
                sample.extend(sorted([str(p) for p in base.glob("llm_features*.csv")])[:25])
            except Exception:
                pass
        sample = sample[:25]
        raise FileNotFoundError(
            f"{provider}: feature file not found for variant={variant_name}, split={split_name}. "
            f"Patterns={patterns}. feature_dir={info['feature_dir']}. Sample existing files={sample}"
        )

    path = matches[0]
    frame = normalize_llm_feature_frame(pd.read_csv(path))

    expected_len = len(llm_train_raw) if split_name == "train" else len(llm_test_raw)
    if len(frame) != expected_len:
        raise ValueError(
            f"{provider}: unexpected length for {path.name}. "
            f"expected={expected_len}, got={len(frame)}"
        )

    return frame, path

def build_provider_feature_bundles(provider: str):
    if provider in PROVIDER_BUNDLE_CACHE:
        return PROVIDER_BUNDLE_CACHE[provider]

    bundles = {}
    path_rows = []

    # 1) llm_only
    llm_train_feat_only, path_train_only = load_provider_variant_features(provider, "llm_only", "train")
    llm_test_feat_only, path_test_only = load_provider_variant_features(provider, "llm_only", "test")

    Xtr_llm_only = pd.concat(
        [meta_Xtr0.reset_index(drop=True), llm_train_feat_only.reset_index(drop=True)],
        axis=1,
    )
    Xte_llm_only_full = pd.concat(
        [meta_Xte0.reset_index(drop=True), llm_test_feat_only.reset_index(drop=True)],
        axis=1,
    )
    Xte_llm_only_typ = Xte_llm_only_full.loc[test_typical_mask].reset_index(drop=True)

    bundles["llm_only"] = {
        "Xtr": Xtr_llm_only,
        "Xte_full": Xte_llm_only_full,
        "Xte_typ": Xte_llm_only_typ,
        "feature_sources": {
            "llm_train": str(path_train_only),
            "llm_test": str(path_test_only),
        },
    }

    path_rows.extend([
        {"provider": provider, "experiment": "llm_only", "split": "train", "path": str(path_train_only)},
        {"provider": provider, "experiment": "llm_only", "split": "test", "path": str(path_test_only)},
    ])

    # 2) cluster_before_llm variants
    for tag, raw_bundle in raw_cluster_variants.items():
        exp_name = f"cluster_before_llm__{tag}"
        llm_train_feat, path_train = load_provider_variant_features(provider, exp_name, "train")
        llm_test_feat, path_test = load_provider_variant_features(provider, exp_name, "test")

        Xtr = pd.concat(
            [
                meta_Xtr0.reset_index(drop=True),
                raw_bundle["train_feat_raw"].reset_index(drop=True),
                llm_train_feat.reset_index(drop=True),
            ],
            axis=1,
        )
        Xte_full = pd.concat(
            [
                meta_Xte0.reset_index(drop=True),
                raw_bundle["test_feat_raw"].reset_index(drop=True),
                llm_test_feat.reset_index(drop=True),
            ],
            axis=1,
        )
        Xte_typ = Xte_full.loc[test_typical_mask].reset_index(drop=True)

        bundles[exp_name] = {
            "Xtr": Xtr,
            "Xte_full": Xte_full,
            "Xte_typ": Xte_typ,
            "feature_sources": {
                "llm_train": str(path_train),
                "llm_test": str(path_test),
            },
        }

        path_rows.extend([
            {"provider": provider, "experiment": exp_name, "split": "train", "path": str(path_train)},
            {"provider": provider, "experiment": exp_name, "split": "test", "path": str(path_test)},
        ])

    # 3) llm_then_cluster variants
    Xtr_llm_space = Xtr_llm_only.copy()
    Xte_llm_space = Xte_llm_only_full.copy()

    cluster_after_scaler = StandardScaler()
    Xtr_llm_scaled = cluster_after_scaler.fit_transform(Xtr_llm_space)
    Xte_llm_scaled = cluster_after_scaler.transform(Xte_llm_space)

    cluster_after_pca = PCA(n_components=min(30, Xtr_llm_space.shape[1]), random_state=seed)
    Xtr_llm_embed = cluster_after_pca.fit_transform(Xtr_llm_scaled)
    Xte_llm_embed = cluster_after_pca.transform(Xte_llm_scaled)

    for cfg in FIXED_CLUSTER_CONFIGS:
        result = fit_clusterer_and_assign(cfg["clusterer"], cfg["params"], Xtr_llm_embed, Xte_llm_embed)
        if result is None:
            raise RuntimeError(f"{provider}: cluster-after-LLM config error: {cfg}")

        aft_tr_labels, aft_te_labels, aft_tr_proba, aft_te_proba, aft_fit_mode = result
        cluster_train_feat, cluster_test_feat = build_cluster_meta_features(
            aft_tr_labels,
            aft_te_labels,
            aft_tr_proba,
            aft_te_proba,
        )

        exp_name = f"llm_then_cluster__{cfg['tag']}"

        Xtr = pd.concat(
            [
                meta_Xtr0.reset_index(drop=True),
                llm_train_feat_only.reset_index(drop=True),
                cluster_train_feat.reset_index(drop=True),
            ],
            axis=1,
        )
        Xte_full = pd.concat(
            [
                meta_Xte0.reset_index(drop=True),
                llm_test_feat_only.reset_index(drop=True),
                cluster_test_feat.reset_index(drop=True),
            ],
            axis=1,
        )
        Xte_typ = Xte_full.loc[test_typical_mask].reset_index(drop=True)

        bundles[exp_name] = {
            "Xtr": Xtr,
            "Xte_full": Xte_full,
            "Xte_typ": Xte_typ,
            "feature_sources": {
                "llm_train": str(path_train_only),
                "llm_test": str(path_test_only),
                "cluster_after_fit_mode": aft_fit_mode,
            },
        }

    PROVIDER_BUNDLE_CACHE[provider] = bundles
    PROVIDER_FEATURE_PATH_LOG[provider] = pd.DataFrame(path_rows).drop_duplicates().reset_index(drop=True)
    return bundles

selected_providers = sorted(selected_best12_per_model_df["provider"].unique().tolist())
bundle_manifest_rows = []
for provider in selected_providers:
    provider_bundles = build_provider_feature_bundles(provider)
    for exp_name, bundle in provider_bundles.items():
        bundle_manifest_rows.append({
            "provider": provider,
            "experiment": exp_name,
            "Xtr_shape": str(bundle["Xtr"].shape),
            "Xte_full_shape": str(bundle["Xte_full"].shape),
            "Xte_typ_shape": str(bundle["Xte_typ"].shape),
            "feature_sources": json.dumps(bundle.get("feature_sources", {}), ensure_ascii=False),
        })

bundle_manifest_df = pd.DataFrame(bundle_manifest_rows)
display(bundle_manifest_df.sort_values(["provider", "experiment"], kind="stable").reset_index(drop=True))
save_df(bundle_manifest_df, "provider_bundle_manifest.csv")

feature_path_log_df = pd.concat(
    [PROVIDER_FEATURE_PATH_LOG[p] for p in selected_providers if p in PROVIDER_FEATURE_PATH_LOG],
    ignore_index=True,
)
save_df(feature_path_log_df, "provider_feature_path_log.csv")


## Load the DL training stack and rebuild the final base pool

Дальше используются те же DL-архитектуры, что и в основном DL-эксперименте. Для каждого выбранного base learner notebook:
- восстанавливает точный feature space;
- делает split-fit для blend-validation;
- делает full-fit на всем train_typical;
- кэширует предсказания для последующего ensembling.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import time, math, gc
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, median_absolute_error

try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
except Exception:
    class _DummyTrialPruned(Exception):
        pass
    class _DummyLogging:
        WARNING = None
        @staticmethod
        def set_verbosity(*args, **kwargs):
            return None
    class _DummyOptuna:
        TrialPruned = _DummyTrialPruned
        logging = _DummyLogging()
    optuna = _DummyOptuna()

try:
    from tabpfn import TabPFNRegressor
    try:
        from tabpfn.constants import ModelVersion
        TABPFN_MODEL_VERSION_ENUM = getattr(ModelVersion, globals().get("TABPFN_FORCE_MODEL_VERSION", "V2_5"), None)
    except Exception:
        ModelVersion = None
        TABPFN_MODEL_VERSION_ENUM = None
    HAS_TABPFN = True
    print("TabPFN доступен ✓")
    print(f"TabPFN HF login status: {globals().get('TABPFN_HF_LOGIN_STATUS', 'unknown')}")
    print(f"TabPFN selected version enum: {TABPFN_MODEL_VERSION_ENUM}")
except ImportError:
    HAS_TABPFN = False
    ModelVersion = None
    TABPFN_MODEL_VERSION_ENUM = None
    print("TabPFN не установлен — установи tabpfn вне ноутбука")

torch.manual_seed(seed)
np.random.seed(seed)

def make_tabpfn_regressor(device: str = "cuda", n_estimators: int = 8):
    if not globals().get("HAS_TABPFN", False):
        raise RuntimeError("TabPFN package is not available.")
    # Для headless runtime принудительно используем Hugging Face-compatible TabPFN-2.5.
    if globals().get("TABPFN_MODEL_VERSION_ENUM", None) is None:
        raise RuntimeError(
            "Current tabpfn build does not expose ModelVersion.V2_5. "
            "Upgrade tabpfn or restart after a clean install."
        )
    reg = TabPFNRegressor.create_default_for_version(globals()["TABPFN_MODEL_VERSION_ENUM"])
    try:
        reg.set_params(device=device, n_estimators=int(n_estimators))
    except Exception:
        # sklearn-style set_params может быть недоступен на некоторых версиях
        try:
            reg.device = device
        except Exception:
            pass
        try:
            reg.n_estimators = int(n_estimators)
        except Exception:
            pass
    return reg


# ═══════════════════════════════════════════════════════════════
#  1. Подготовка данных
# ═══════════════════════════════════════════════════════════════

val_cut = int(len(Xtrain) * 0.8)

X_dl_tr = Xtrain.iloc[:val_cut].values.astype(np.float32)
y_dl_tr = ytrain.iloc[:val_cut].values.astype(np.float32)
X_dl_va = Xtrain.iloc[val_cut:].values.astype(np.float32)
y_dl_va = ytrain.iloc[val_cut:].values.astype(np.float32)

sc = StandardScaler()
X_dl_tr = sc.fit_transform(X_dl_tr)
X_dl_va = sc.transform(X_dl_va)
X_dl_te = sc.transform(Xtest.values.astype(np.float32))

sc_full = StandardScaler()
X_full_s = sc_full.fit_transform(Xtrain.values.astype(np.float32))
X_te_full_s = sc_full.transform(Xtest.values.astype(np.float32))

for arr in [X_dl_tr, X_dl_va, X_dl_te, X_full_s, X_te_full_s]:
    np.nan_to_num(arr, copy=False)

X_tr_t = torch.from_numpy(X_dl_tr)
y_tr_t = torch.from_numpy(y_dl_tr).unsqueeze(1)
X_va_t = torch.from_numpy(X_dl_va)
y_va_t = torch.from_numpy(y_dl_va).unsqueeze(1)
X_te_t = torch.from_numpy(X_dl_te)
X_full_t = torch.from_numpy(X_full_s)
y_full_t = torch.from_numpy(ytrain.values.astype(np.float32)).unsqueeze(1)
X_te_full_t = torch.from_numpy(X_te_full_s)

device = torch.device(
    "mps" if torch.backends.mps.is_available() else
    "cuda" if torch.cuda.is_available() else "cpu"
)
INPUT_DIM = X_dl_tr.shape[1]
print(f"Device: {device},  Input dim: {INPUT_DIM}")
print(f"Train: {X_dl_tr.shape}, Val: {X_dl_va.shape}, Test: {X_dl_te.shape}\n")


# ═══════════════════════════════════════════════════════════════
#  2. АРХИТЕКТУРЫ  (24 шт.)
# ═══════════════════════════════════════════════════════════════

# ── 2.1  MLP ─────────────────────────────────────────────────

class MLP(nn.Module):
    def __init__(self, d_in, hidden_dims, dropout=0.3):
        super().__init__()
        layers = []
        prev = d_in
        for h in hidden_dims:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h),
                       nn.SiLU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x)


# ── 2.2  ResMLP ──────────────────────────────────────────────

class ResBlock(nn.Module):
    def __init__(self, dim, dropout=0.3):
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(dim, dim), nn.BatchNorm1d(dim), nn.SiLU(), nn.Dropout(dropout),
            nn.Linear(dim, dim), nn.BatchNorm1d(dim),
        )
        self.act = nn.SiLU()
    def forward(self, x):
        return self.act(x + self.block(x))

class ResMLP(nn.Module):
    def __init__(self, d_in, hidden=256, n_blocks=3, dropout=0.3):
        super().__init__()
        self.proj = nn.Sequential(nn.Linear(d_in, hidden), nn.BatchNorm1d(hidden), nn.SiLU())
        self.blocks = nn.Sequential(*[ResBlock(hidden, dropout) for _ in range(n_blocks)])
        self.head = nn.Linear(hidden, 1)
    def forward(self, x):
        return self.head(self.blocks(self.proj(x)))


# ── 2.3  SNN ─────────────────────────────────────────────────

class SNN(nn.Module):
    def __init__(self, d_in, hidden_dims=[256, 128], alpha_dropout=0.1):
        super().__init__()
        layers = []
        prev = d_in
        for h in hidden_dims:
            layers += [nn.Linear(prev, h), nn.SELU(), nn.AlphaDropout(alpha_dropout)]
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
        self._init_weights()
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_in', nonlinearity='linear')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
    def forward(self, x):
        return self.net(x)


# ── 2.4  GatedMLP ───────────────────────────────────────────

class GatedBlock(nn.Module):
    def __init__(self, d_in, d_out, dropout):
        super().__init__()
        self.linear = nn.Linear(d_in, d_out * 2)
        self.bn = nn.BatchNorm1d(d_out)
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        h = self.linear(x)
        h1, h2 = h.chunk(2, dim=-1)
        return self.drop(self.bn(h1 * torch.sigmoid(h2)))

class GatedMLP(nn.Module):
    def __init__(self, d_in, hidden=256, n_blocks=3, dropout=0.3):
        super().__init__()
        blocks = [GatedBlock(d_in, hidden, dropout)]
        for _ in range(n_blocks - 1):
            blocks.append(GatedBlock(hidden, hidden, dropout))
        self.blocks = nn.Sequential(*blocks)
        self.head = nn.Linear(hidden, 1)
    def forward(self, x):
        return self.head(self.blocks(x))


# ── 2.5  GANDALF ─────────────────────────────────────────────

class GANDALF(nn.Module):
    def __init__(self, d_in, hidden=256, n_blocks=3, dropout=0.3):
        super().__init__()
        self.gate_fc = nn.Linear(d_in, d_in)
        layers = [nn.Linear(d_in, hidden), nn.BatchNorm1d(hidden), nn.SiLU(), nn.Dropout(dropout)]
        for _ in range(n_blocks - 1):
            layers += [nn.Linear(hidden, hidden), nn.BatchNorm1d(hidden), nn.SiLU(), nn.Dropout(dropout)]
        self.dnn = nn.Sequential(*layers)
        self.head = nn.Linear(hidden, 1)
    def forward(self, x):
        gate = torch.sigmoid(self.gate_fc(x))
        return self.head(self.dnn(x * gate))


# ── 2.6  DAE-MLP ────────────────────────────────────────────

class DAEMLP(nn.Module):
    def __init__(self, d_in, hidden=256, latent=64, noise=0.3, dropout=0.3):
        super().__init__()
        self.noise = noise
        self.encoder = nn.Sequential(
            nn.Linear(d_in, hidden), nn.BatchNorm1d(hidden), nn.SiLU(), nn.Dropout(dropout),
            nn.Linear(hidden, latent), nn.BatchNorm1d(latent), nn.SiLU(),
        )
        self.decoder = nn.Sequential(nn.Linear(latent, hidden), nn.SiLU(), nn.Linear(hidden, d_in))
        self.reg_head = nn.Sequential(nn.Dropout(dropout), nn.Linear(latent, 32), nn.SiLU(), nn.Linear(32, 1))
        self.aux_loss = torch.tensor(0.0)
    def forward(self, x):
        x_in = x * (torch.rand_like(x) > self.noise).float() if self.training and self.noise > 0 else x
        z = self.encoder(x_in)
        if self.training:
            self.aux_loss = F.mse_loss(self.decoder(z), x)
        return self.reg_head(z)


# ── 2.7  CNN1D ───────────────────────────────────────────────

class CNN1D(nn.Module):
    def __init__(self, d_in, channels=[64, 128, 64], ks=5, dropout=0.3):
        super().__init__()
        layers = []
        in_ch = 1
        for ch in channels:
            layers += [nn.Conv1d(in_ch, ch, ks, padding=ks // 2),
                       nn.BatchNorm1d(ch), nn.SiLU(), nn.Dropout(dropout)]
            in_ch = ch
        self.conv = nn.Sequential(*layers)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.head = nn.Sequential(nn.Linear(channels[-1], 64), nn.SiLU(), nn.Dropout(dropout), nn.Linear(64, 1))
    def forward(self, x):
        x = x.unsqueeze(1)
        x = self.pool(self.conv(x))
        return self.head(x.squeeze(-1))


# ── 2.8  BiGRU ───────────────────────────────────────────────

class BiGRU(nn.Module):
    def __init__(self, d_in, hidden=64, n_layers=2, dropout=0.3):
        super().__init__()
        self.gru = nn.GRU(input_size=1, hidden_size=hidden, num_layers=n_layers,
                          batch_first=True, bidirectional=True,
                          dropout=dropout if n_layers > 1 else 0)
        self.head = nn.Sequential(nn.Linear(hidden * 2, 64), nn.SiLU(), nn.Dropout(dropout), nn.Linear(64, 1))
    def forward(self, x):
        _, h = self.gru(x.unsqueeze(-1))
        return self.head(torch.cat([h[-2], h[-1]], dim=1))


# ── 2.9  FT-Transformer ─────────────────────────────────────

class FTTransformer(nn.Module):
    def __init__(self, d_in, d_model=32, n_heads=4, n_layers=2, dropout=0.2):
        super().__init__()
        self.feat_emb = nn.Linear(1, d_model)
        self.cls = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)
        self.pos = nn.Parameter(torch.randn(1, d_in + 1, d_model) * 0.02)
        enc = nn.TransformerEncoderLayer(d_model=d_model, nhead=n_heads,
              dim_feedforward=d_model * 4, dropout=dropout, activation="gelu", batch_first=True)
        self.transformer = nn.TransformerEncoder(enc, num_layers=n_layers)
        self.head = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, 1))
    def forward(self, x):
        B = x.shape[0]
        tok = self.feat_emb(x.unsqueeze(-1))
        tok = torch.cat([self.cls.expand(B, -1, -1), tok], dim=1) + self.pos
        return self.head(self.transformer(tok)[:, 0])


# ── 2.10  TabTransformer ────────────────────────────────────

class TabTransformer(nn.Module):
    def __init__(self, d_in, d_model=32, n_heads=4, n_layers=2, mlp_hidden=128, dropout=0.2):
        super().__init__()
        self.feat_emb = nn.Linear(1, d_model)
        self.col_emb = nn.Parameter(torch.randn(1, d_in, d_model) * 0.02)
        enc = nn.TransformerEncoderLayer(d_model=d_model, nhead=n_heads,
              dim_feedforward=d_model * 4, dropout=dropout, activation="gelu", batch_first=True)
        self.transformer = nn.TransformerEncoder(enc, num_layers=n_layers)
        self.head = nn.Sequential(
            nn.Linear(d_in * d_model + d_in, mlp_hidden), nn.SiLU(), nn.Dropout(dropout),
            nn.Linear(mlp_hidden, 1))
    def forward(self, x):
        tok = self.feat_emb(x.unsqueeze(-1)) + self.col_emb
        out = self.transformer(tok)
        pooled = out.reshape(out.size(0), -1)
        return self.head(torch.cat([pooled, x], dim=1))


# ── 2.11  SAINT ──────────────────────────────────────────────

class SAINTBlock(nn.Module):
    def __init__(self, d_model, n_heads, dropout):
        super().__init__()
        self.sa_norm = nn.LayerNorm(d_model)
        self.self_attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.ia_norm = nn.LayerNorm(d_model)
        self.inter_attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.ffn = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, d_model * 4),
                                 nn.GELU(), nn.Dropout(dropout), nn.Linear(d_model * 4, d_model), nn.Dropout(dropout))
    def forward(self, x):
        h = self.sa_norm(x)
        h, _ = self.self_attn(h, h, h)
        x = x + h
        B, F_, D = x.shape
        if B > 1 and self.training:
            xt = x.permute(1, 0, 2)
            h2 = self.ia_norm(xt)
            h2, _ = self.inter_attn(h2, h2, h2)
            x = (xt + h2).permute(1, 0, 2)
        return x + self.ffn(x)

class SAINT(nn.Module):
    def __init__(self, d_in, d_model=32, n_heads=4, n_layers=2, dropout=0.2):
        super().__init__()
        self.feat_emb = nn.Linear(1, d_model)
        self.cls = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)
        self.pos = nn.Parameter(torch.randn(1, d_in + 1, d_model) * 0.02)
        self.blocks = nn.ModuleList([SAINTBlock(d_model, n_heads, dropout) for _ in range(n_layers)])
        self.head = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, 1))
    def forward(self, x):
        B = x.shape[0]
        tok = self.feat_emb(x.unsqueeze(-1))
        tok = torch.cat([self.cls.expand(B, -1, -1), tok], dim=1) + self.pos
        for blk in self.blocks:
            tok = blk(tok)
        return self.head(tok[:, 0])


# ── 2.12  AutoInt ────────────────────────────────────────────

class AutoIntLayer(nn.Module):
    def __init__(self, d_model, n_heads, dropout):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.norm = nn.LayerNorm(d_model)
        self.W_res = nn.Linear(d_model, d_model, bias=False)
    def forward(self, x):
        h, _ = self.attn(x, x, x)
        return F.relu(self.norm(h + self.W_res(x)))

class AutoInt(nn.Module):
    def __init__(self, d_in, d_model=32, n_heads=4, n_layers=3, dropout=0.2):
        super().__init__()
        self.feat_emb = nn.Linear(1, d_model)
        self.layers = nn.ModuleList([AutoIntLayer(d_model, n_heads, dropout) for _ in range(n_layers)])
        self.head = nn.Sequential(nn.Linear(d_in * d_model, 256), nn.ReLU(), nn.Dropout(dropout), nn.Linear(256, 1))
    def forward(self, x):
        tok = self.feat_emb(x.unsqueeze(-1))
        for layer in self.layers:
            tok = layer(tok)
        return self.head(tok.reshape(tok.size(0), -1))


# ── 2.13  Trompt ─────────────────────────────────────────────

class TromptLayer(nn.Module):
    def __init__(self, d_in, d_model, n_heads, dropout):
        super().__init__()
        self.prompt = nn.Parameter(torch.randn(1, d_in, d_model) * 0.02)
        self.attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.norm1 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(nn.Linear(d_model, d_model * 2), nn.GELU(), nn.Dropout(dropout),
                                 nn.Linear(d_model * 2, d_model))
        self.norm2 = nn.LayerNorm(d_model)
        self.gate = nn.Linear(d_model, 1)
    def forward(self, feat_emb, x_raw):
        B = feat_emb.shape[0]
        prompted = feat_emb + self.prompt.expand(B, -1, -1)
        h, _ = self.attn(prompted, prompted, prompted)
        h = self.norm1(prompted + h)
        h = self.norm2(h + self.ffn(h))
        weights = torch.softmax(self.gate(h).squeeze(-1), dim=1)
        return h, weights * x_raw

class Trompt(nn.Module):
    def __init__(self, d_in, d_model=32, n_heads=4, n_layers=3, dropout=0.2):
        super().__init__()
        self.feat_emb = nn.Linear(1, d_model)
        self.layers = nn.ModuleList([TromptLayer(d_in, d_model, n_heads, dropout) for _ in range(n_layers)])
        self.head = nn.Sequential(nn.Linear(d_in, 128), nn.ReLU(), nn.Dropout(dropout), nn.Linear(128, 1))
    def forward(self, x):
        tok = self.feat_emb(x.unsqueeze(-1))
        preds = []
        for layer in self.layers:
            tok, lp = layer(tok, x)
            preds.append(lp)
        return self.head(torch.stack(preds).mean(0))


# ── 2.14  ExcelFormer ───────────────────────────────────────

class ExcelFormer(nn.Module):
    def __init__(self, d_in, d_model=32, n_heads=4, n_layers=2, dropout=0.2):
        super().__init__()
        self.feat_emb = nn.Linear(1, d_model)
        self.importance = nn.Parameter(torch.zeros(d_in))
        self.cls = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)
        self.pos = nn.Parameter(torch.randn(1, d_in + 1, d_model) * 0.02)
        enc = nn.TransformerEncoderLayer(d_model=d_model, nhead=n_heads,
              dim_feedforward=d_model * 4, dropout=dropout, activation="gelu", batch_first=True)
        self.transformer = nn.TransformerEncoder(enc, num_layers=n_layers)
        self.head = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, 1))
    def forward(self, x):
        B = x.shape[0]
        mask = torch.sigmoid(self.importance)
        x_masked = x * mask
        tok = self.feat_emb(x_masked.unsqueeze(-1))
        tok = torch.cat([self.cls.expand(B, -1, -1), tok], dim=1) + self.pos
        return self.head(self.transformer(tok)[:, 0])


# ── 2.15  ARM-Net ────────────────────────────────────────────

class ARMNet(nn.Module):
    def __init__(self, d_in, n_exp=64, hidden=128, order=2, dropout=0.2):
        super().__init__()
        self.order = order
        self.exp_layers = nn.ModuleList([nn.Linear(d_in, n_exp) for _ in range(order)])
        self.gate = nn.Sequential(nn.Linear(d_in, n_exp * order), nn.Sigmoid())
        self.head = nn.Sequential(
            nn.BatchNorm1d(n_exp * order), nn.Linear(n_exp * order, hidden),
            nn.SiLU(), nn.Dropout(dropout), nn.Linear(hidden, 1))
    def forward(self, x):
        interactions = []
        for i, layer in enumerate(self.exp_layers):
            h = layer(x)
            if i > 0:
                h = h * interactions[-1]
            interactions.append(F.softplus(h))
        cat = torch.cat(interactions, dim=1)
        return self.head(cat * self.gate(x))


# ── 2.16  NODE ───────────────────────────────────────────────

class NODE(nn.Module):
    def __init__(self, d_in, n_trees=32, depth=4, dropout=0.0):
        super().__init__()
        self.n_trees = n_trees
        self.depth = depth
        n_leaves = 2 ** depth
        self.feat_weights = nn.Parameter(torch.randn(n_trees, depth, d_in) * 0.01)
        self.thresholds = nn.Parameter(torch.zeros(n_trees, depth))
        self.leaf_values = nn.Parameter(torch.randn(n_trees, n_leaves) * 0.01)
        self.drop = nn.Dropout(dropout)
        patterns = torch.zeros(n_leaves, depth)
        for i in range(n_leaves):
            for d in range(depth):
                patterns[i, d] = (i >> (depth - 1 - d)) & 1
        self.register_buffer("patterns", patterns)
    def forward(self, x):
        choices = torch.sigmoid(torch.einsum('bj,tdj->btd', x, self.feat_weights) - self.thresholds)
        c = choices.unsqueeze(2)
        p = self.patterns.unsqueeze(0).unsqueeze(0)
        leaf_probs = (c * p + (1 - c) * (1 - p)).prod(dim=-1)
        tree_out = (leaf_probs * self.leaf_values.unsqueeze(0)).sum(dim=-1)
        return self.drop(tree_out).mean(dim=-1, keepdim=True)


# ── 2.17  GRANDE ─────────────────────────────────────────────

class GRANDE(nn.Module):
    def __init__(self, d_in, n_trees=32, depth=4, dropout=0.0):
        super().__init__()
        self.n_trees = n_trees
        self.depth = depth
        n_leaves = 2 ** depth
        self.feat_logits = nn.Parameter(torch.randn(n_trees, depth, d_in) * 0.1)
        self.thresholds = nn.Parameter(torch.zeros(n_trees, depth))
        self.leaf_values = nn.Parameter(torch.randn(n_trees, n_leaves) * 0.01)
        self.tree_weights = nn.Parameter(torch.ones(n_trees) / n_trees)
        self.drop = nn.Dropout(dropout)
        patterns = torch.zeros(n_leaves, depth)
        for i in range(n_leaves):
            for d in range(depth):
                patterns[i, d] = (i >> (depth - 1 - d)) & 1
        self.register_buffer("patterns", patterns)
    def forward(self, x):
        feat_sel = F.softmax(self.feat_logits, dim=-1)
        proj = torch.einsum('bj,tdj->btd', x, feat_sel)
        choices = torch.sigmoid(10.0 * (proj - self.thresholds))
        c = choices.unsqueeze(2)
        p = self.patterns.unsqueeze(0).unsqueeze(0)
        leaf_probs = (c * p + (1 - c) * (1 - p)).prod(dim=-1)
        tree_out = (leaf_probs * self.leaf_values.unsqueeze(0)).sum(dim=-1)
        w = F.softmax(self.tree_weights, dim=0)
        return self.drop((tree_out * w).sum(dim=-1, keepdim=True))


# ── 2.18  Net-DNF ────────────────────────────────────────────

class NetDNF(nn.Module):
    def __init__(self, d_in, n_formulas=64, n_conjuncts=6, dropout=0.2):
        super().__init__()
        self.feat_sel = nn.Parameter(torch.randn(n_formulas, n_conjuncts, d_in) * 0.1)
        self.thresholds = nn.Parameter(torch.zeros(n_formulas, n_conjuncts))
        self.formula_w = nn.Linear(n_formulas, 1)
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        sel = torch.sigmoid(self.feat_sel)
        proj = torch.einsum('bj,fcj->bfc', x, sel)
        conjuncts = torch.sigmoid(proj - self.thresholds)
        formulas = conjuncts.prod(dim=-1)
        return self.formula_w(self.drop(formulas))


# ── 2.19  TabNet (simplified) ───────────────────────────────

class TabNet(nn.Module):
    def __init__(self, d_in, n_steps=3, n_d=64, n_a=64, relax=1.5, dropout=0.1):
        super().__init__()
        self.n_steps = n_steps
        self.relax = relax
        self.bn_init = nn.BatchNorm1d(d_in)
        self.shared_fc = nn.Linear(d_in, n_d + n_a)
        self.step_fcs = nn.ModuleList([nn.Linear(d_in, n_d + n_a) for _ in range(n_steps)])
        self.attn_fcs = nn.ModuleList([nn.Sequential(nn.Linear(n_a, d_in), nn.BatchNorm1d(d_in)) for _ in range(n_steps)])
        self.head = nn.Linear(n_d, 1)
        self.n_d = n_d
        self.drop = nn.Dropout(dropout)
        self.aux_loss = torch.tensor(0.0)
    def forward(self, x):
        x = self.bn_init(x)
        prior_scales = torch.ones(x.shape[0], x.shape[1], device=x.device)
        agg = torch.zeros(x.shape[0], self.n_d, device=x.device)
        entropy_loss = 0.0
        for i in range(self.n_steps):
            h = self.shared_fc(x * prior_scales) + self.step_fcs[i](x * prior_scales)
            h_d, h_a = h[:, :self.n_d], h[:, self.n_d:]
            h_d = F.silu(h_d)
            agg = agg + self.drop(h_d)
            attn = self.attn_fcs[i](h_a)
            attn = F.softmax(attn * prior_scales, dim=-1)
            prior_scales = prior_scales * (self.relax - attn)
            entropy_loss += (-attn * torch.log(attn + 1e-15)).mean()
        if self.training:
            self.aux_loss = entropy_loss / self.n_steps
        return self.head(agg)


# ── 2.20  TabR ───────────────────────────────────────────────

class TabR(nn.Module):
    def __init__(self, d_in, hidden=256, n_layers=3, k=32, dropout=0.3):
        super().__init__()
        self.k = k
        layers = []
        prev = d_in
        for i in range(n_layers):
            h = hidden if i == 0 else hidden // 2
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.SiLU(), nn.Dropout(dropout)]
            prev = h
        self.encoder = nn.Sequential(*layers)
        self.latent_dim = prev
        self.retrieval_head = nn.Sequential(nn.Linear(prev * 2, 128), nn.SiLU(), nn.Dropout(dropout), nn.Linear(128, 1))
        self.direct_head = nn.Linear(prev, 1)
        self._store_z = None
        self._store_y = None
    def build_store(self, x_train, y_train):
        self.eval()
        with torch.no_grad():
            self._store_z = self.encoder(x_train)
            self._store_y = y_train
    def forward(self, x):
        z = self.encoder(x)
        if self._store_z is not None and not self.training:
            dists = torch.cdist(z, self._store_z)
            _, idx = dists.topk(self.k, largest=False)
            nbr_z = self._store_z[idx]
            sim = -dists.gather(1, idx)
            attn = torch.softmax(sim, dim=1).unsqueeze(-1)
            context = (attn * nbr_z).sum(dim=1)
            return self.retrieval_head(torch.cat([z, context], dim=1))
        return self.direct_head(z)


# ── 2.21  GrowNet stages ────────────────────────────────────

class GrowNetStage(nn.Module):
    def __init__(self, d_in, hidden=32, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_in + 1, hidden), nn.BatchNorm1d(hidden), nn.SiLU(), nn.Dropout(dropout),
            nn.Linear(hidden, hidden), nn.SiLU(), nn.Linear(hidden, 1))
    def forward(self, x, prev_pred):
        return self.net(torch.cat([x, prev_pred], dim=1))


# ── 2.22  SwitchTab ──────────────────────────────────────────

class SwitchTab(nn.Module):
    def __init__(self, d_in, d_enc=128, d_mutual=64, d_salient=64, dropout=0.3):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(d_in, d_enc), nn.BatchNorm1d(d_enc), nn.SiLU(), nn.Dropout(dropout))
        self.mutual_proj = nn.Linear(d_enc, d_mutual)
        self.salient_proj = nn.Linear(d_enc, d_salient)
        self.decoder = nn.Sequential(nn.Linear(d_mutual + d_salient, d_in))
        self.head = nn.Sequential(nn.Linear(d_mutual + d_salient, 64), nn.SiLU(), nn.Dropout(dropout), nn.Linear(64, 1))
        self.aux_loss = torch.tensor(0.0)
    def forward(self, x):
        h = self.encoder(x)
        mutual = self.mutual_proj(h)
        salient = self.salient_proj(h)
        rep = torch.cat([mutual, salient], dim=1)
        if self.training:
            self.aux_loss = F.mse_loss(self.decoder(rep), x)
        return self.head(rep)


# ── 2.23  PTaRL ──────────────────────────────────────────────

class PTaRL(nn.Module):
    def __init__(self, d_in, n_prototypes=16, d_latent=128, hidden=256, dropout=0.3):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(d_in, hidden), nn.BatchNorm1d(hidden), nn.SiLU(), nn.Dropout(dropout),
            nn.Linear(hidden, d_latent), nn.BatchNorm1d(d_latent), nn.SiLU())
        self.prototypes = nn.Parameter(torch.randn(n_prototypes, d_latent) * 0.1)
        self.head = nn.Sequential(nn.Linear(d_latent * 2, 64), nn.SiLU(), nn.Dropout(dropout), nn.Linear(64, 1))
    def forward(self, x):
        z = self.encoder(x)
        sim = F.cosine_similarity(z.unsqueeze(1), self.prototypes.unsqueeze(0), dim=-1)
        attn = F.softmax(sim * 10, dim=1)
        proto_rep = (attn.unsqueeze(-1) * self.prototypes.unsqueeze(0)).sum(dim=1)
        return self.head(torch.cat([z, proto_rep], dim=1))


# ── 2.24  DCN v2 ────────────────────────────────────────────

class CrossLayer(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.W = nn.Linear(d, d, bias=False)
        self.b = nn.Parameter(torch.zeros(d))
    def forward(self, x0, x):
        return x0 * self.W(x) + self.b + x

class DCNv2(nn.Module):
    def __init__(self, d_in, n_cross=3, hidden=256, dropout=0.3):
        super().__init__()
        self.cross_layers = nn.ModuleList([CrossLayer(d_in) for _ in range(n_cross)])
        self.deep = nn.Sequential(
            nn.Linear(d_in, hidden), nn.BatchNorm1d(hidden), nn.SiLU(), nn.Dropout(dropout),
            nn.Linear(hidden, hidden // 2), nn.BatchNorm1d(hidden // 2), nn.SiLU(), nn.Dropout(dropout))
        self.head = nn.Linear(d_in + hidden // 2, 1)
    def forward(self, x):
        x_cross = x
        for cl in self.cross_layers:
            x_cross = cl(x, x_cross)
        return self.head(torch.cat([x_cross, self.deep(x)], dim=1))


# ═══════════════════════════════════════════════════════════════
#  3. ФУНКЦИИ ОБУЧЕНИЯ
# ═══════════════════════════════════════════════════════════════

def train_model(model, epochs=300, lr=1e-3, wd=1e-4, bs=256,
                patience=30, aux_w=0.0, trial=None):
    """Обучение с early stopping. Возвращает (model, val_mae, epochs_used)."""
    model = model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", patience=10, factor=0.5, min_lr=1e-6)
    loss_fn = nn.L1Loss()
    ds = TensorDataset(X_tr_t, y_tr_t)
    dl = DataLoader(ds, batch_size=bs, shuffle=True)
    X_v, y_v = X_va_t.to(device), y_va_t.to(device)

    best_val, best_state, wait = float("inf"), None, 0
    for ep in range(1, epochs + 1):
        model.train()
        for xb, yb in dl:
            xb, yb = xb.to(device), yb.to(device)
            pred = model(xb)
            loss = loss_fn(pred, yb)
            if aux_w > 0 and hasattr(model, "aux_loss"):
                loss = loss + aux_w * model.aux_loss
            opt.zero_grad(); loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            v_mae = loss_fn(model(X_v), y_v).item()
        sched.step(v_mae)
        if v_mae < best_val:
            best_val = v_mae
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                break
        if trial is not None:
            trial.report(v_mae, ep)
            if trial.should_prune():
                raise optuna.TrialPruned()
    model.load_state_dict(best_state)
    return model, best_val, ep


def train_grownet(n_stages=5, hidden=32, lr=0.01, wd=1e-4,
                  step_size=0.1, bs=256, patience=30, dropout=0.1, trial=None):
    """Gradient boosting с NN base learners. Возвращает (stages, val_mae, total_epochs)."""
    stages = []
    X_v, y_v = X_va_t.to(device), y_va_t.to(device)
    ds = TensorDataset(X_tr_t, y_tr_t)
    dl = DataLoader(ds, batch_size=bs, shuffle=True)
    loss_fn = nn.L1Loss()

    # Предсказание ансамбля на train/val
    def ensemble_pred(x, stgs):
        p = torch.zeros(x.size(0), 1, device=x.device)
        for s in stgs:
            s.eval()
            p = p + step_size * s(x, p.detach())
        return p

    best_val, best_n = float("inf"), 0
    total_ep = 0
    for stage_idx in range(n_stages):
        model = GrowNetStage(INPUT_DIM, hidden, dropout).to(device)
        opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
        sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", patience=10, factor=0.5, min_lr=1e-6)
        best_stage_val, best_stage_state, wait = float("inf"), None, 0
        for ep in range(1, 201):
            model.train()
            for xb, yb in dl:
                xb, yb = xb.to(device), yb.to(device)
                with torch.no_grad():
                    prev = ensemble_pred(xb, stages)
                correction = model(xb, prev)
                loss = loss_fn(prev + step_size * correction, yb)
                opt.zero_grad(); loss.backward(); opt.step()
            model.eval()
            with torch.no_grad():
                prev_v = ensemble_pred(X_v, stages)
                v_pred = prev_v + step_size * model(X_v, prev_v)
                v_mae = loss_fn(v_pred, y_v).item()
            sched.step(v_mae)
            if v_mae < best_stage_val:
                best_stage_val = v_mae
                best_stage_state = {k: v.clone() for k, v in model.state_dict().items()}
                wait = 0
            else:
                wait += 1
                if wait >= patience:
                    break
            total_ep += 1
        model.load_state_dict(best_stage_state)
        stages.append(model)
        # Проверяем ансамбль
        with torch.no_grad():
            ens_val = loss_fn(ensemble_pred(X_v, stages), y_v).item()
        if ens_val < best_val:
            best_val = ens_val
            best_n = len(stages)
        if trial is not None:
            trial.report(ens_val, stage_idx)
            if trial.should_prune():
                raise optuna.TrialPruned()
    stages = stages[:best_n]
    return stages, best_val, total_ep


def eval_on_test(model, name=""):
    """Evaluate model on holdout test."""
    model.eval()
    with torch.no_grad():
        pred = model(X_te_t.to(device)).cpu().numpy().flatten()
    mae = mean_absolute_error(ytest, pred)
    rmse = np.sqrt(mean_squared_error(ytest, pred))
    mdae = median_absolute_error(ytest, pred)
    return mae, rmse, mdae


def eval_grownet_on_test(stages, step_size=0.1):
    """Evaluate GrowNet ensemble on holdout test."""
    X_t = X_te_t.to(device)
    p = torch.zeros(X_t.size(0), 1, device=device)
    for s in stages:
        s.eval()
        with torch.no_grad():
            p = p + step_size * s(X_t, p)
    pred = p.cpu().numpy().flatten()
    mae = mean_absolute_error(ytest, pred)
    rmse = np.sqrt(mean_squared_error(ytest, pred))
    mdae = median_absolute_error(ytest, pred)
    return mae, rmse, mdae


print("24 архитектуры + train_model + train_grownet определены.")


In [ ]:
# Дополнительные тензоры/функции для полного holdout (full holdout),
# чтобы не ломать исходные train/eval-хелперы из исходного ноутбука.

X_dl_te_full = sc.transform(Xtest_full.values.astype(np.float32))
np.nan_to_num(X_dl_te_full, copy=False)
X_te_stress_t = torch.from_numpy(X_dl_te_full)

X_te_fullscale_stress = sc_full.transform(Xtest_full.values.astype(np.float32))
np.nan_to_num(X_te_fullscale_stress, copy=False)
X_te_fullscale_stress_t = torch.from_numpy(X_te_fullscale_stress)

def clear_device_cache():
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()
    elif device.type == "mps":
        torch.mps.empty_cache()

def calc_reg_metrics(y_true, pred):
    return {
        "MAE": float(mean_absolute_error(y_true, pred)),
        "RMSE": float(np.sqrt(mean_squared_error(y_true, pred))),
        "MdAE": float(median_absolute_error(y_true, pred)),
    }

def eval_on_typical_holdout(model):
    model.eval()
    with torch.no_grad():
        pred = model(X_te_t.to(device)).cpu().numpy().flatten()
    return calc_reg_metrics(ytest_typical, pred)

def eval_on_full_holdout(model):
    model.eval()
    with torch.no_grad():
        pred = model(X_te_stress_t.to(device)).cpu().numpy().flatten()
    return calc_reg_metrics(ytest_full, pred)

def eval_grownet_on_typical_holdout(stages, step_size=0.1):
    X_t = X_te_t.to(device)
    p = torch.zeros(X_t.size(0), 1, device=device)
    for s in stages:
        s.eval()
        with torch.no_grad():
            p = p + step_size * s(X_t, p)
    pred = p.cpu().numpy().flatten()
    return calc_reg_metrics(ytest_typical, pred)

def eval_grownet_on_full_holdout(stages, step_size=0.1):
    X_t = X_te_stress_t.to(device)
    p = torch.zeros(X_t.size(0), 1, device=device)
    for s in stages:
        s.eval()
        with torch.no_grad():
            p = p + step_size * s(X_t, p)
    pred = p.cpu().numpy().flatten()
    return calc_reg_metrics(ytest_full, pred)

def eval_tabpfn_holdouts(tabpfn_model, use_full_train_scaler=False):
    if use_full_train_scaler:
        X_typ = X_te_full_t.numpy()
        X_full = X_te_fullscale_stress_t.numpy()
    else:
        X_typ = X_te_t.numpy()
        X_full = X_te_stress_t.numpy()
    pred_typ = tabpfn_model.predict(X_typ)
    pred_full = tabpfn_model.predict(X_full)
    return calc_reg_metrics(ytest_typical, pred_typ), calc_reg_metrics(ytest_full, pred_full)

def train_model_fulltrain(model, epochs=100, lr=1e-3, wd=1e-4, bs=256, aux_w=0.0):
    model = model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(epochs, 1), eta_min=1e-6)
    loss_fn = nn.L1Loss()
    ds = TensorDataset(X_full_t, y_full_t)
    dl = DataLoader(ds, batch_size=bs, shuffle=True)
    for ep in range(epochs):
        model.train()
        for xb, yb in dl:
            xb, yb = xb.to(device), yb.to(device)
            pred = model(xb)
            loss = loss_fn(pred, yb)
            if aux_w > 0 and hasattr(model, "aux_loss"):
                loss = loss + aux_w * model.aux_loss
            opt.zero_grad()
            loss.backward()
            opt.step()
        sched.step()
    return model

def eval_fulltrain_model_typical(model):
    model.eval()
    with torch.no_grad():
        pred = model(X_te_full_t.to(device)).cpu().numpy().flatten()
    return calc_reg_metrics(ytest_typical, pred)

def eval_fulltrain_model_full(model):
    model.eval()
    with torch.no_grad():
        pred = model(X_te_fullscale_stress_t.to(device)).cpu().numpy().flatten()
    return calc_reg_metrics(ytest_full, pred)

def train_grownet_fulltrain(n_stages=5, hidden=32, lr=0.01, wd=1e-4,
                            step_size=0.1, bs=256, epochs_per_stage=50, dropout=0.1):
    stages = []
    ds = TensorDataset(X_full_t, y_full_t)
    dl = DataLoader(ds, batch_size=bs, shuffle=True)
    loss_fn = nn.L1Loss()

    def ensemble_pred(x, stgs):
        p = torch.zeros(x.size(0), 1, device=x.device)
        for s in stgs:
            s.eval()
            p = p + step_size * s(x, p.detach())
        return p

    for stage_idx in range(n_stages):
        model = GrowNetStage(INPUT_DIM, hidden, dropout).to(device)
        opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(epochs_per_stage, 1), eta_min=1e-6)

        for ep in range(epochs_per_stage):
            model.train()
            for xb, yb in dl:
                xb, yb = xb.to(device), yb.to(device)
                with torch.no_grad():
                    prev = ensemble_pred(xb, stages)
                correction = model(xb, prev)
                loss = loss_fn(prev + step_size * correction, yb)
                opt.zero_grad()
                loss.backward()
                opt.step()
            sched.step()

        stages.append(model)
    return stages

def eval_fulltrain_grownet_typical(stages, step_size=0.1):
    X_t = X_te_full_t.to(device)
    p = torch.zeros(X_t.size(0), 1, device=device)
    for s in stages:
        s.eval()
        with torch.no_grad():
            p = p + step_size * s(X_t, p)
    pred = p.cpu().numpy().flatten()
    return calc_reg_metrics(ytest_typical, pred)

def eval_fulltrain_grownet_full(stages, step_size=0.1):
    X_t = X_te_fullscale_stress_t.to(device)
    p = torch.zeros(X_t.size(0), 1, device=device)
    for s in stages:
        s.eval()
        with torch.no_grad():
            p = p + step_size * s(X_t, p)
    pred = p.cpu().numpy().flatten()
    return calc_reg_metrics(ytest_full, pred)

print("Full-holdout helpers defined.")


def setup_dl_for_variant(Xtr_aug, ytr, Xte_typ_aug, yte_typ, Xte_full_aug, yte_full):
    """Перестраивает scaler / тензоры / INPUT_DIM под текущее feature space.
    Делает глобальные переменные совместимыми с train_model / train_grownet / fulltrain-хелперами."""
    global X_dl_tr, y_dl_tr, X_dl_va, y_dl_va, X_dl_te, X_full_s, X_te_full_s
    global X_tr_t, y_tr_t, X_va_t, y_va_t, X_te_t, X_te_stress_t
    global X_full_t, y_full_t, X_te_full_t, X_te_fullscale_stress_t
    global INPUT_DIM, sc, sc_full
    global Xtrain, ytrain, Xtest, ytest, ytest_typical, ytest_full

    Xtrain = pd.DataFrame(Xtr_aug).reset_index(drop=True).copy()
    ytrain = pd.Series(ytr).reset_index(drop=True).copy()
    Xtest = pd.DataFrame(Xte_typ_aug).reset_index(drop=True).copy()
    ytest = pd.Series(yte_typ).reset_index(drop=True).copy()
    ytest_typical = ytest.copy()
    ytest_full = pd.Series(yte_full).reset_index(drop=True).copy()

    val_cut = int(len(Xtrain) * (1 - DL_INNER_VAL_FRAC))
    if val_cut <= 0 or val_cut >= len(Xtrain):
        raise ValueError(f"Некорректный DL_INNER_VAL_FRAC={DL_INNER_VAL_FRAC} для len(Xtrain)={len(Xtrain)}")

    Xtr_arr = Xtrain.values.astype(np.float32)
    ytr_arr = ytrain.values.astype(np.float32)
    Xte_typ_arr = pd.DataFrame(Xte_typ_aug).reset_index(drop=True).values.astype(np.float32)
    Xte_full_arr = pd.DataFrame(Xte_full_aug).reset_index(drop=True).values.astype(np.float32)

    X_dl_tr = Xtr_arr[:val_cut]
    y_dl_tr = ytr_arr[:val_cut]
    X_dl_va = Xtr_arr[val_cut:]
    y_dl_va = ytr_arr[val_cut:]

    sc = StandardScaler()
    X_dl_tr = sc.fit_transform(X_dl_tr)
    X_dl_va = sc.transform(X_dl_va)
    X_dl_te = sc.transform(Xte_typ_arr)
    X_dl_te_full = sc.transform(Xte_full_arr)

    sc_full = StandardScaler()
    X_full_s = sc_full.fit_transform(Xtr_arr)
    X_te_full_s = sc_full.transform(Xte_typ_arr)
    X_te_fullscale_stress = sc_full.transform(Xte_full_arr)

    for arr in [X_dl_tr, X_dl_va, X_dl_te, X_dl_te_full, X_full_s, X_te_full_s, X_te_fullscale_stress]:
        np.nan_to_num(arr, copy=False)

    X_tr_t = torch.from_numpy(X_dl_tr)
    y_tr_t = torch.from_numpy(y_dl_tr).unsqueeze(1)
    X_va_t = torch.from_numpy(X_dl_va)
    y_va_t = torch.from_numpy(y_dl_va).unsqueeze(1)
    X_te_t = torch.from_numpy(X_dl_te)
    X_te_stress_t = torch.from_numpy(X_dl_te_full)
    X_full_t = torch.from_numpy(X_full_s)
    y_full_t = torch.from_numpy(ytr_arr).unsqueeze(1)
    X_te_full_t = torch.from_numpy(X_te_full_s)
    X_te_fullscale_stress_t = torch.from_numpy(X_te_fullscale_stress)

    INPUT_DIM = X_dl_tr.shape[1]
    print(f"  setup_dl_for_variant: INPUT_DIM={INPUT_DIM}, train={X_dl_tr.shape}, val={X_dl_va.shape}, test_typ={X_dl_te.shape}, test_full={X_dl_te_full.shape}")


In [ ]:
def build_arch_from_params(arch_name: str, bp: dict):
    if arch_name == "MLP":
        dims = [max(int(bp["first_hidden"]) // (2 ** i), 32) for i in range(int(bp["n_layers"]))]
        return MLP(INPUT_DIM, dims, bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "ResMLP":
        return ResMLP(INPUT_DIM, bp["hidden"], bp["n_blocks"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "SNN":
        dims = [max(int(bp["first_hidden"]) // (2 ** i), 32) for i in range(int(bp["n_layers"]))]
        return SNN(INPUT_DIM, dims, bp["alpha_dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "GatedMLP":
        return GatedMLP(INPUT_DIM, bp["hidden"], bp["n_blocks"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "GANDALF":
        return GANDALF(INPUT_DIM, bp["hidden"], bp["n_blocks"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "DAE-MLP":
        return DAEMLP(INPUT_DIM, bp["hidden"], bp["latent"], bp["noise"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "CNN1D":
        nc = int(bp["n_conv"]); bc = int(bp["base_ch"])
        chs = [bc * (2 ** i) for i in range(nc // 2 + 1)]
        chs += [bc * (2 ** i) for i in range(nc // 2 - 1, -1, -1)]
        chs = chs[:nc]
        return CNN1D(INPUT_DIM, chs, bp["ks"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "BiGRU":
        return BiGRU(INPUT_DIM, bp["hidden"], bp["n_layers"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "FT-Transformer":
        return FTTransformer(INPUT_DIM, bp["d_model"], bp["n_heads"], bp["n_layers"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "TabTransformer":
        return TabTransformer(INPUT_DIM, bp["d_model"], bp["n_heads"], bp["n_layers"], bp["mlp_hidden"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "SAINT":
        return SAINT(INPUT_DIM, bp["d_model"], bp["n_heads"], bp["n_layers"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "AutoInt":
        return AutoInt(INPUT_DIM, bp["d_model"], bp["n_heads"], bp["n_layers"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "Trompt":
        return Trompt(INPUT_DIM, bp["d_model"], bp["n_heads"], bp["n_layers"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "ExcelFormer":
        return ExcelFormer(INPUT_DIM, bp["d_model"], bp["n_heads"], bp["n_layers"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "ARM-Net":
        return ARMNet(INPUT_DIM, bp["n_exp"], bp["hidden"], bp["order"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "NODE":
        return NODE(INPUT_DIM, bp["n_trees"], bp["depth"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "GRANDE":
        return GRANDE(INPUT_DIM, bp["n_trees"], bp["depth"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "Net-DNF":
        return NetDNF(INPUT_DIM, bp["n_formulas"], bp["n_conjuncts"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "TabNet":
        return TabNet(INPUT_DIM, bp["n_steps"], bp["n_d"], bp["n_a"], bp["relax"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "TabR":
        return TabR(INPUT_DIM, bp["hidden"], bp["n_layers"], bp["k"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "SwitchTab":
        return SwitchTab(INPUT_DIM, bp["d_enc"], bp["d_mutual"], bp["d_salient"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "PTaRL":
        return PTaRL(INPUT_DIM, bp["n_prototypes"], bp["d_latent"], bp["hidden"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "DCNv2":
        return DCNv2(INPUT_DIM, bp["n_cross"], bp["hidden"], bp["dropout"]), bp.get("aux_w", 0.0)
    else:
        raise KeyError(f"Unknown architecture: {arch_name}")

def serialize_params(params: dict):
    clean = {}
    for k, v in params.items():
        if isinstance(v, (np.integer,)):
            clean[k] = int(v)
        elif isinstance(v, (np.floating,)):
            clean[k] = float(v)
        else:
            clean[k] = v
    return clean


def parse_best_params(raw):
    if isinstance(raw, dict):
        bp = raw
    elif raw is None:
        bp = {}
    else:
        try:
            if pd.isna(raw):
                bp = {}
            else:
                bp = raw
        except Exception:
            bp = raw

    if isinstance(bp, dict):
        return serialize_params(bp)

    s = str(bp).strip()
    if not s or s.lower() in {"nan", "none", "null"}:
        return {}

    try:
        return serialize_params(json.loads(s))
    except Exception:
        pass

    try:
        return serialize_params(ast.literal_eval(s))
    except Exception:
        pass

    s2 = re.sub(r"\bNaN\b", "None", s, flags=re.IGNORECASE)
    s2 = re.sub(r"\bnull\b", "None", s2, flags=re.IGNORECASE)
    s2 = re.sub(r"\btrue\b", "True", s2, flags=re.IGNORECASE)
    s2 = re.sub(r"\bfalse\b", "False", s2, flags=re.IGNORECASE)
    return serialize_params(ast.literal_eval(s2))

blend_split_idx = int(len(meta_Xtr0) * BLEND_TRAIN_FRAC)
if blend_split_idx <= 0 or blend_split_idx >= len(meta_Xtr0):
    raise ValueError("Некорректный BLEND_TRAIN_FRAC.")

base_pool_df = selected_best12_per_model_df.copy()
if KEEP_ONLY_POSITIVE_DELTA and "delta_typical" in base_pool_df.columns:
    base_pool_df = base_pool_df[base_pool_df["delta_typical"] > 0].copy()

if MAX_BASE_LEARNERS is not None:
    base_pool_df = (
        base_pool_df
        .sort_values(["cv_MAE_internal", "holdout_typical_MAE", "holdout_full_MAE"], kind="stable")
        .head(int(MAX_BASE_LEARNERS))
        .copy()
    )

base_pool_df["best_params_parsed"] = base_pool_df["best_params"].apply(parse_best_params)
base_pool_df["epochs_hint"] = (
    pd.to_numeric(base_pool_df.get("epochs_used", np.nan), errors="coerce")
    .fillna(pd.to_numeric(base_pool_df.get("epochs", np.nan), errors="coerce"))
    .fillna(100)
    .astype(int)
)

display(
    base_pool_df[
        ["base_id", "model", "provider", "scheme_family", "experiment", "cv_MAE_internal", "holdout_typical_MAE", "holdout_full_MAE", "epochs_hint"]
    ]
    .sort_values(["cv_MAE_internal", "holdout_typical_MAE", "holdout_full_MAE"], kind="stable")
    .reset_index(drop=True)
)
save_df(base_pool_df, "selected_base_pool.csv")

# Shared cache for TabPFN model weights/checkpoints
TABPFN_CACHE_DIR = (DRIVE_PROJECT_DIR / "tabpfn_model_cache") if (REMOTE_RUNTIME and USE_REMOTE_STORAGE) else (ARTIFACT_ROOT / "tabpfn_model_cache")
TABPFN_CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("TABPFN_MODEL_CACHE_DIR", str(TABPFN_CACHE_DIR))
print("TABPFN_MODEL_CACHE_DIR =", os.environ.get("TABPFN_MODEL_CACHE_DIR"))

def prepare_tabpfn_checkpoint_if_needed():
    if "TabPFN" not in set(base_pool_df["model"].tolist()):
        print("TabPFN not selected in final base pool -> checkpoint preparation skipped.")
        return

    try:
        from huggingface_hub import hf_hub_download
    except Exception as e:
        print(f"[WARN] huggingface_hub unavailable: {e}")
        return

    hf_token = (
        os.environ.get("HF_TOKEN")
        or os.environ.get("HUGGING_FACE_HUB_TOKEN")
        or os.environ.get("HUGGINGFACEHUB_API_TOKEN")
        or os.environ.get("HUGGINGFACE_TOKEN")
    )

    if not hf_token:
        print("HF token не задан. TabPFN-блок пропущен до настройки окружения.")
        return

    target = TABPFN_CACHE_DIR / "tabpfn-v2.5-regressor-v2.5_default.ckpt"
    if target.exists():
        print("TabPFN checkpoint already exists:", target)
        os.environ["TABPFN_MODEL_CACHE_DIR"] = str(TABPFN_CACHE_DIR)
        return

    downloaded = hf_hub_download(
        repo_id="Prior-Labs/tabpfn_2_5",
        filename="tabpfn-v2.5-regressor-v2.5_default.ckpt",
        token=hf_token,
        force_download=False,
    )
    shutil.copyfile(downloaded, target)
    os.environ["TABPFN_MODEL_CACHE_DIR"] = str(TABPFN_CACHE_DIR)
    print("Prepared TabPFN checkpoint:", target)

prepare_tabpfn_checkpoint_if_needed()

PRED_CACHE_DIR = RUN_DIR / "pred_cache"
for sub in ["blend_val", "fullfit_test_typical", "fullfit_test_full"]:
    (PRED_CACHE_DIR / sub).mkdir(parents=True, exist_ok=True)


In [ ]:

# ------------------------------------------------------------
# Refit selected DL bases and cache their predictions
# ------------------------------------------------------------
def _pred_path(kind: str, base_id: str) -> Path:
    return PRED_CACHE_DIR / kind / f"{sanitize_name(base_id)}.npy"

def _save_pred(kind: str, base_id: str, arr):
    np.save(_pred_path(kind, base_id), np.asarray(arr, dtype=np.float32))

def _load_pred(kind: str, base_id: str):
    path = _pred_path(kind, base_id)
    if not path.exists():
        return None
    return np.load(path)

def _have_all_cached(base_id: str) -> bool:
    needed = [
        _pred_path("blend_val", base_id),
        _pred_path("fullfit_test_typical", base_id),
        _pred_path("fullfit_test_full", base_id),
    ]
    return all(p.exists() for p in needed)

def _torch_pred(model, X_tensor):
    model.eval()
    with torch.no_grad():
        pred = model(X_tensor.to(device)).cpu().numpy().reshape(-1)
    return np.clip(pred, 0, None)

def _grownet_pred(stages, X_tensor, step_size=0.1):
    X_t = X_tensor.to(device)
    p = torch.zeros(X_t.size(0), 1, device=device)
    for s in stages:
        s.eval()
        with torch.no_grad():
            p = p + step_size * s(X_t, p)
    pred = p.cpu().numpy().reshape(-1)
    return np.clip(pred, 0, None)

def resolve_experiment_bundle_from_row(row: pd.Series):
    provider = row["provider"]
    experiment_name = row["experiment"]
    bundles = build_provider_feature_bundles(provider)
    if experiment_name not in bundles:
        raise KeyError(f"{provider}: experiment bundle not found: {experiment_name}")
    return bundles[experiment_name]

def _fit_dl_arch_for_current_globals(arch: str, best_params: dict, epochs_hint: int):
    if arch == "TabPFN":
        if not globals().get("HAS_TABPFN", False):
            raise RuntimeError("TabPFN выбран, но пакет tabpfn недоступен.")
        pfn_device = "cuda" if torch.cuda.is_available() else "cpu"
        max_rows = int(best_params.get("max_rows", 3000))
        n_estimators = int(best_params.get("n_estimators", 8))

        tabpfn = make_tabpfn_regressor(device=pfn_device, n_estimators=n_estimators)
        X_pfn_tr = X_dl_tr.copy()
        y_pfn_tr = y_dl_tr.copy()
        if len(X_pfn_tr) > max_rows:
            idx = np.random.RandomState(SEED).choice(len(X_pfn_tr), max_rows, replace=False)
            X_pfn_tr = X_pfn_tr[idx]
            y_pfn_tr = y_pfn_tr[idx]
        tabpfn.fit(X_pfn_tr, y_pfn_tr)
        pred_target = np.clip(tabpfn.predict(X_dl_te), 0, None)
        return {"kind": "tabpfn", "pred_target": pred_target}

    elif arch == "GrowNet":
        stages, _, _ = train_grownet(
            n_stages=int(best_params["n_stages"]),
            hidden=int(best_params["hidden"]),
            lr=float(best_params["lr"]),
            wd=float(best_params["wd"]),
            step_size=float(best_params["step_size"]),
            bs=int(best_params["bs"]),
            patience=30,
            dropout=float(best_params["dropout"]),
        )
        pred_target = _grownet_pred(stages, X_te_t, step_size=float(best_params["step_size"]))
        return {"kind": "grownet", "pred_target": pred_target}

    else:
        model, aux_w = build_arch_from_params(arch, best_params)
        model, _, _ = train_model(
            model,
            epochs=max(1, int(epochs_hint)),
            lr=float(best_params["lr"]),
            wd=float(best_params["wd"]),
            bs=int(best_params["bs"]),
            patience=30,
            aux_w=float(aux_w),
        )
        if arch == "TabR":
            model.build_store(X_tr_t.to(device), y_tr_t.to(device))
        pred_target = _torch_pred(model, X_te_t)
        return {"kind": "torch", "pred_target": pred_target}

def _fullfit_dl_arch_for_current_globals(arch: str, best_params: dict, epochs_hint: int):
    if arch == "TabPFN":
        if not globals().get("HAS_TABPFN", False):
            raise RuntimeError("TabPFN выбран, но пакет tabpfn недоступен.")
        pfn_device = "cuda" if torch.cuda.is_available() else "cpu"
        max_rows = int(best_params.get("max_rows", 3000))
        n_estimators = int(best_params.get("n_estimators", 8))

        tabpfn_full = make_tabpfn_regressor(device=pfn_device, n_estimators=n_estimators)
        X_pfn_full = X_full_s.copy()
        y_pfn_full = ytrain.values.astype(np.float32)
        if len(X_pfn_full) > max_rows:
            idx = np.random.RandomState(SEED).choice(len(X_pfn_full), max_rows, replace=False)
            X_pfn_full = X_pfn_full[idx]
            y_pfn_full = y_pfn_full[idx]
        tabpfn_full.fit(X_pfn_full, y_pfn_full)
        pred_typ = np.clip(tabpfn_full.predict(X_te_full_t.numpy()), 0, None)
        pred_full = np.clip(tabpfn_full.predict(X_te_fullscale_stress_t.numpy()), 0, None)
        return pred_typ, pred_full

    elif arch == "GrowNet":
        n_stages = int(best_params["n_stages"])
        epochs_per_stage = max(20, math.ceil(max(1, int(epochs_hint)) / max(n_stages, 1)))
        stages_full = train_grownet_fulltrain(
            n_stages=n_stages,
            hidden=int(best_params["hidden"]),
            lr=float(best_params["lr"]),
            wd=float(best_params["wd"]),
            step_size=float(best_params["step_size"]),
            bs=int(best_params["bs"]),
            epochs_per_stage=epochs_per_stage,
            dropout=float(best_params["dropout"]),
        )
        pred_typ = _grownet_pred(stages_full, X_te_full_t, step_size=float(best_params["step_size"]))
        pred_full = _grownet_pred(stages_full, X_te_fullscale_stress_t, step_size=float(best_params["step_size"]))
        return pred_typ, pred_full

    else:
        full_model, aux_w_full = build_arch_from_params(arch, best_params)
        full_model = train_model_fulltrain(
            full_model,
            epochs=max(1, int(epochs_hint)),
            lr=float(best_params["lr"]),
            wd=float(best_params["wd"]),
            bs=int(best_params["bs"]),
            aux_w=float(aux_w_full),
        )
        if arch == "TabR":
            full_model.build_store(X_full_t.to(device), y_full_t.to(device))
        pred_typ = _torch_pred(full_model, X_te_full_t)
        pred_full = _torch_pred(full_model, X_te_fullscale_stress_t)
        return pred_typ, pred_full

def fit_and_predict_base_row(row: pd.Series):
    base_id = row["base_id"]
    arch = row["model"]
    best_params = row["best_params_parsed"]
    epochs_hint = max(1, int(row.get("epochs_hint", 100)))

    if RESUME_IF_PREDICTIONS_EXIST and not FORCE_REFRESH_BASE_PREDICTIONS and _have_all_cached(base_id):
        return {
            "base_id": base_id,
            "status": "cached",
            "model": arch,
            "provider": row["provider"],
            "scheme_family": row["scheme_family"],
            "experiment": row["experiment"],
            "elapsed_sec": 0.0,
        }

    bundle = resolve_experiment_bundle_from_row(row)
    Xtr_all = bundle["Xtr"].reset_index(drop=True).copy()
    Xte_typ = bundle["Xte_typ"].reset_index(drop=True).copy()
    Xte_full = bundle["Xte_full"].reset_index(drop=True).copy()

    X_blend_tr = Xtr_all.iloc[:blend_split_idx].reset_index(drop=True).copy()
    X_blend_val = Xtr_all.iloc[blend_split_idx:].reset_index(drop=True).copy()
    y_blend_tr = meta_ytr0.iloc[:blend_split_idx].reset_index(drop=True).copy()
    y_blend_val_local = meta_ytr0.iloc[blend_split_idx:].reset_index(drop=True).copy()

    t0 = time.time()

    # Split-fit -> predict on external blend-val
    setup_dl_for_variant(
        X_blend_tr, y_blend_tr,
        X_blend_val, y_blend_val_local,
        X_blend_val, y_blend_val_local,
    )
    seed_everything(SEED)
    split_out = _fit_dl_arch_for_current_globals(arch, best_params, epochs_hint)
    pred_blend_val = split_out["pred_target"]
    clear_device_cache()

    # Full-fit -> predict on holdout
    setup_dl_for_variant(
        Xtr_all, meta_ytr0,
        Xte_typ, meta_yte_typ0,
        Xte_full, meta_yte0,
    )
    seed_everything(SEED)
    pred_test_typ, pred_test_full = _fullfit_dl_arch_for_current_globals(arch, best_params, epochs_hint)
    clear_device_cache()

    _save_pred("blend_val", base_id, pred_blend_val)
    _save_pred("fullfit_test_typical", base_id, pred_test_typ)
    _save_pred("fullfit_test_full", base_id, pred_test_full)

    return {
        "base_id": base_id,
        "status": "rebuilt",
        "model": arch,
        "provider": row["provider"],
        "scheme_family": row["scheme_family"],
        "experiment": row["experiment"],
        "elapsed_sec": round(time.time() - t0, 2),
        "blend_val_MAE": mae_metric(y_blend_val_local, pred_blend_val),
        "holdout_typical_MAE": mae_metric(meta_yte_typ0, pred_test_typ),
        "holdout_full_MAE": mae_metric(meta_yte0, pred_test_full),
    }

def build_base_prediction_matrices(pool_frame: pd.DataFrame):
    rows = []
    for _, row in pool_frame.iterrows():
        try:
            out = fit_and_predict_base_row(row)
            rows.append(out)
        except Exception as e:
            rows.append({
                "base_id": row["base_id"],
                "status": "error",
                "model": row["model"],
                "provider": row["provider"],
                "scheme_family": row["scheme_family"],
                "experiment": row["experiment"],
                "error": repr(e),
            })
            print(f"Ошибка: {row['base_id']} -> {e}")
            clear_device_cache()
    return pd.DataFrame(rows)

def load_prediction_frame(kind: str, base_ids):
    cols = {}
    for base_id in base_ids:
        arr = _load_pred(kind, base_id)
        if arr is not None:
            cols[base_id] = arr
    return pd.DataFrame(cols)

model_cache_log_df = build_base_prediction_matrices(base_pool_df)
_sort_cols = [c for c in ["status", "holdout_typical_MAE"] if c in model_cache_log_df.columns]
if _sort_cols:
    display(model_cache_log_df.sort_values(_sort_cols, kind="stable").reset_index(drop=True))
else:
    display(model_cache_log_df.reset_index(drop=True))
save_df(model_cache_log_df, "model_cache_log.csv")

selected_base_ids = base_pool_df["base_id"].tolist()
val_pred_df = load_prediction_frame("blend_val", selected_base_ids)
test_typ_fullfit_pred_df = load_prediction_frame("fullfit_test_typical", selected_base_ids)
test_full_fullfit_pred_df = load_prediction_frame("fullfit_test_full", selected_base_ids)

available_base_ids = [
    base_id for base_id in selected_base_ids
    if base_id in val_pred_df.columns
    and base_id in test_typ_fullfit_pred_df.columns
    and base_id in test_full_fullfit_pred_df.columns
]

val_pred_df = val_pred_df[available_base_ids].copy()
test_typ_fullfit_pred_df = test_typ_fullfit_pred_df[available_base_ids].copy()
test_full_fullfit_pred_df = test_full_fullfit_pred_df[available_base_ids].copy()

if len(available_base_ids) < 2:
    raise RuntimeError("Для полноценного ансамблирования нужно хотя бы 2 успешно восстановленных base learners.")

y_blend_val = meta_ytr0.iloc[blend_split_idx:].reset_index(drop=True).to_numpy(dtype=float)
y_test_typ = meta_yte_typ0.to_numpy(dtype=float)
y_test_full = meta_yte0.to_numpy(dtype=float)

val_pred_df.to_csv(RUN_DIR / "base_pred_blend_val.csv", index=False)
test_typ_fullfit_pred_df.to_csv(RUN_DIR / "base_pred_fullfit_test_typical.csv", index=False)
test_full_fullfit_pred_df.to_csv(RUN_DIR / "base_pred_fullfit_test_full.csv", index=False)

base_meta_df = (
    base_pool_df[
        ["base_id", "model", "provider", "scheme_family", "experiment", "cv_MAE_internal", "holdout_typical_MAE", "holdout_full_MAE", "epochs_hint"]
    ]
    .set_index("base_id")
    .loc[available_base_ids]
    .reset_index()
)

display(base_meta_df)
save_df(base_meta_df, "available_base_meta.csv")

print("Available base learners:", len(available_base_ids))
print(available_base_ids)


## Search over blending / stacking specifications

In [ ]:
# ------------------------------------------------------------
# Ensemble helpers
# ------------------------------------------------------------
def get_base_leaderboard_from_predictions(pred_df: pd.DataFrame, y_true):
    rows = []
    for col in pred_df.columns:
        rows.append({
            "base_id": col,
            "MAE": mae_metric(y_true, pred_df[col].values),
            "RMSE": rmse_metric(y_true, pred_df[col].values),
            "MdAE": mdae_metric(y_true, pred_df[col].values),
        })
    out = pd.DataFrame(rows).sort_values(["MAE", "RMSE", "MdAE"], kind="stable").reset_index(drop=True)
    meta = base_pool_df[["base_id", "model", "provider", "scheme_family", "experiment"]].drop_duplicates()
    return out.merge(meta, on="base_id", how="left")

def aggregate_predictions(pred_matrix: np.ndarray, method: str, trim_frac: float = 0.1):
    pred_matrix = np.asarray(pred_matrix, dtype=float)
    if pred_matrix.ndim == 1:
        return clip_pred(pred_matrix)
    if method == "mean":
        out = pred_matrix.mean(axis=1)
    elif method == "median":
        out = np.median(pred_matrix, axis=1)
    elif method == "trimmed_mean":
        m = pred_matrix.shape[1]
        k = int(np.floor(m * trim_frac))
        if k <= 0 or 2 * k >= m:
            out = pred_matrix.mean(axis=1)
        else:
            sorted_mat = np.sort(pred_matrix, axis=1)
            out = sorted_mat[:, k:m-k].mean(axis=1)
    else:
        raise KeyError(method)
    return clip_pred(out)

def normalize_weights(w):
    w = np.asarray(w, dtype=float)
    w = np.where(np.isfinite(w), w, 0.0)
    w = np.clip(w, 0, None)
    s = w.sum()
    if s <= 0:
        return np.ones_like(w) / len(w)
    return w / s

def weights_from_errors(pred_fit: pd.DataFrame, y_fit, scheme: str, temperature: float = 1.0, power: float = 1.0):
    errs = np.array([mae_metric(y_fit, pred_fit[c].values) for c in pred_fit.columns], dtype=float)
    if scheme == "equal":
        w = np.ones(len(errs), dtype=float)
    elif scheme == "inverse_mae":
        w = 1.0 / np.maximum(errs, 1e-9)
    elif scheme == "inverse_mae_sq":
        w = 1.0 / np.maximum(errs, 1e-9) ** 2
    elif scheme == "softmax_mae":
        z = -errs / max(float(temperature), 1e-6)
        z = z - z.max()
        w = np.exp(z)
    elif scheme == "rank_power":
        order = np.argsort(errs)
        ranks = np.empty_like(order)
        ranks[order] = np.arange(1, len(errs) + 1)
        w = 1.0 / np.power(ranks, power)
    else:
        raise KeyError(scheme)
    return normalize_weights(w)

def fit_weighted_blender(pred_fit: pd.DataFrame, y_fit, scheme: str, **kwargs):
    X = pred_fit.values.astype(float)
    if scheme in {"equal", "inverse_mae", "inverse_mae_sq", "softmax_mae", "rank_power"}:
        w = weights_from_errors(pred_fit, y_fit, scheme, **kwargs)
        return {"kind": "weights", "weights": w}
    elif scheme == "nnls":
        w, _ = nnls(X, np.asarray(y_fit, dtype=float))
        w = normalize_weights(w)
        return {"kind": "weights", "weights": w}
    elif scheme == "simplex_mae":
        n = X.shape[1]
        x0 = np.ones(n, dtype=float) / n
        bounds = [(0.0, 1.0)] * n
        cons = {"type": "eq", "fun": lambda w: np.sum(w) - 1.0}

        def objective(w):
            return mae_metric(y_fit, X @ w)

        res = minimize(
            objective,
            x0=x0,
            method="SLSQP",
            bounds=bounds,
            constraints=[cons],
            options={"maxiter": 500, "ftol": 1e-8},
        )
        w = normalize_weights(res.x if res.success else x0)
        return {"kind": "weights", "weights": w, "optimizer_success": bool(res.success)}
    else:
        raise KeyError(scheme)

def predict_weighted_blender(fitted, pred_df: pd.DataFrame):
    w = np.asarray(fitted["weights"], dtype=float)
    return clip_pred(pred_df.values @ w)

def build_stacker(stacker_name: str):
    if stacker_name == "linear":
        return LinearRegression()
    elif stacker_name == "positive_linear":
        return LinearRegression(positive=True)
    elif stacker_name == "ridge":
        return Ridge(alpha=1.0)
    elif stacker_name == "lasso":
        return Lasso(alpha=1e-3, max_iter=20000, random_state=seed)
    elif stacker_name == "elastic":
        return ElasticNet(alpha=1e-3, l1_ratio=0.5, max_iter=20000, random_state=seed)
    elif stacker_name == "bayes":
        return BayesianRidge()
    elif stacker_name == "huber":
        return HuberRegressor(max_iter=2000, epsilon=1.35)
    elif stacker_name == "quantile":
        return QuantileRegressor(quantile=0.5, alpha=1e-4, solver="highs")
    elif stacker_name == "gbr":
        return GradientBoostingRegressor(
            loss="absolute_error",
            random_state=seed,
            n_estimators=300,
            learning_rate=0.05,
            max_depth=2,
            min_samples_leaf=5,
            min_samples_split=4,
            subsample=0.9,
        )
    elif stacker_name == "rf":
        return RandomForestRegressor(
            n_estimators=500,
            random_state=seed,
            min_samples_leaf=2,
            n_jobs=-1,
        )
    elif stacker_name == "etr":
        return ExtraTreesRegressor(
            n_estimators=500,
            random_state=seed,
            min_samples_leaf=2,
            n_jobs=-1,
        )
    elif stacker_name == "xgb":
        if not HAS_XGB_FOR_STACKING:
            raise RuntimeError("xgboost недоступен для stacker xgb")
        return XGBRegressor(
            objective="reg:absoluteerror",
            eval_metric="mae",
            n_estimators=500,
            learning_rate=0.03,
            max_depth=2,
            min_child_weight=3,
            subsample=0.9,
            colsample_bytree=0.9,
            reg_alpha=0.0,
            reg_lambda=1.0,
            tree_method="hist",
            n_jobs=-1,
            random_state=seed,
            verbosity=0,
        )
    else:
        raise KeyError(stacker_name)

def fit_stacker(pred_fit: pd.DataFrame, y_fit, stacker_name: str):
    model = build_stacker(stacker_name)
    model.fit(pred_fit.values.astype(float), np.asarray(y_fit, dtype=float))
    return {"kind": "stacker", "model": model, "stacker_name": stacker_name}

def predict_stacker(fitted, pred_df: pd.DataFrame):
    return clip_pred(fitted["model"].predict(pred_df.values.astype(float)))

def fit_pair_weight_grid(pred_fit: pd.DataFrame, y_fit):
    if pred_fit.shape[1] != 2:
        raise ValueError("pair_grid expects exactly 2 models")
    best = None
    a = pred_fit.iloc[:, 0].values.astype(float)
    b = pred_fit.iloc[:, 1].values.astype(float)
    for w in PAIR_WEIGHT_GRID:
        pred = clip_pred(w * a + (1.0 - w) * b)
        score = mae_metric(y_fit, pred)
        if best is None or score < best["selection_MAE"]:
            best = {"kind": "pair_grid", "weight_first": float(w), "selection_MAE": score}
    return best

def predict_pair_weight_grid(fitted, pred_df: pd.DataFrame):
    w = float(fitted["weight_first"])
    a = pred_df.iloc[:, 0].values.astype(float)
    b = pred_df.iloc[:, 1].values.astype(float)
    return clip_pred(w * a + (1.0 - w) * b)

def top_models_by_fit_mae(pred_fit_df: pd.DataFrame, y_fit):
    fit_rank_df = get_base_leaderboard_from_predictions(pred_fit_df, y_fit)
    ranked_models = fit_rank_df["base_id"].tolist()
    return ranked_models, fit_rank_df

def spec_tag(spec):
    models_part = "__".join(spec["models"])
    return f'{spec["family"]}::{spec["name"]}::{models_part}'

def evaluate_spec_on_selection(spec, pred_fit_df, y_fit, pred_sel_df, y_sel):
    models = spec["models"]
    fit_sub = pred_fit_df[models].copy()
    sel_sub = pred_sel_df[models].copy()

    family = spec["family"]
    name = spec["name"]

    if family == "single":
        pred_sel = clip_pred(sel_sub.iloc[:, 0].values)
        out = {"family": family, "name": name, "models": models, "n_models": len(models)}
    elif family == "aggregate":
        pred_sel = aggregate_predictions(sel_sub.values, spec["agg_method"], spec.get("trim_frac", 0.1))
        out = {"family": family, "name": name, "models": models, "n_models": len(models)}
    elif family == "weighted":
        fitted = fit_weighted_blender(fit_sub, y_fit, spec["weight_scheme"], **spec.get("weight_kwargs", {}))
        pred_sel = predict_weighted_blender(fitted, sel_sub)
        out = {
            "family": family,
            "name": name,
            "models": models,
            "n_models": len(models),
            "weights": fitted.get("weights", []).tolist() if isinstance(fitted.get("weights", []), np.ndarray) else fitted.get("weights", []),
        }
    elif family == "pair_grid":
        fitted = fit_pair_weight_grid(fit_sub, y_fit)
        pred_sel = predict_pair_weight_grid(fitted, sel_sub)
        out = {
            "family": family,
            "name": name,
            "models": models,
            "n_models": len(models),
            "weight_first": fitted["weight_first"],
        }
    elif family == "stacker":
        fitted = fit_stacker(fit_sub, y_fit, spec["stacker_name"])
        pred_sel = predict_stacker(fitted, sel_sub)
        out = {
            "family": family,
            "name": name,
            "models": models,
            "n_models": len(models),
            "stacker_name": spec["stacker_name"],
        }
    else:
        raise KeyError(family)

    sel_metrics = score_predictions(y_sel, pred_sel)
    out["selection_MAE"] = sel_metrics["MAE"]
    out["selection_RMSE"] = sel_metrics["RMSE"]
    out["selection_MdAE"] = sel_metrics["MdAE"]
    out["tag"] = spec_tag(spec)
    out["spec"] = spec
    return out

def refit_and_eval_spec(spec, pred_val_df, y_val, pred_test_typ_df, y_test_typ, pred_test_full_df, y_test_full):
    models = spec["models"]
    val_sub = pred_val_df[models].copy()
    test_typ_sub = pred_test_typ_df[models].copy()
    test_full_sub = pred_test_full_df[models].copy()

    family = spec["family"]
    name = spec["name"]

    if family == "single":
        pred_typ = clip_pred(test_typ_sub.iloc[:, 0].values)
        pred_full = clip_pred(test_full_sub.iloc[:, 0].values)
        out = {"family": family, "name": name, "models": models, "n_models": len(models)}
    elif family == "aggregate":
        pred_typ = aggregate_predictions(test_typ_sub.values, spec["agg_method"], spec.get("trim_frac", 0.1))
        pred_full = aggregate_predictions(test_full_sub.values, spec["agg_method"], spec.get("trim_frac", 0.1))
        out = {"family": family, "name": name, "models": models, "n_models": len(models)}
    elif family == "weighted":
        fitted = fit_weighted_blender(val_sub, y_val, spec["weight_scheme"], **spec.get("weight_kwargs", {}))
        pred_typ = predict_weighted_blender(fitted, test_typ_sub)
        pred_full = predict_weighted_blender(fitted, test_full_sub)
        out = {
            "family": family,
            "name": name,
            "models": models,
            "n_models": len(models),
            "weights": fitted.get("weights", []).tolist() if isinstance(fitted.get("weights", []), np.ndarray) else fitted.get("weights", []),
        }
    elif family == "pair_grid":
        fitted = fit_pair_weight_grid(val_sub, y_val)
        pred_typ = predict_pair_weight_grid(fitted, test_typ_sub)
        pred_full = predict_pair_weight_grid(fitted, test_full_sub)
        out = {
            "family": family,
            "name": name,
            "models": models,
            "n_models": len(models),
            "weight_first": fitted["weight_first"],
            "weights_like": [fitted["weight_first"], 1.0 - fitted["weight_first"]],
        }
    elif family == "stacker":
        fitted = fit_stacker(val_sub, y_val, spec["stacker_name"])
        pred_typ = predict_stacker(fitted, test_typ_sub)
        pred_full = predict_stacker(fitted, test_full_sub)
        weights_like = None
        if hasattr(fitted["model"], "coef_"):
            coef = np.ravel(getattr(fitted["model"], "coef_"))
            if len(coef) == len(models):
                weights_like = coef.tolist()
        out = {
            "family": family,
            "name": name,
            "models": models,
            "n_models": len(models),
            "weights_like": weights_like,
        }
    else:
        raise KeyError(family)

    typ_metrics = score_predictions(y_test_typ, pred_typ)
    full_metrics = score_predictions(y_test_full, pred_full)

    out["tag"] = spec_tag(spec)
    out["MAE_typical"] = typ_metrics["MAE"]
    out["RMSE_typical"] = typ_metrics["RMSE"]
    out["MdAE_typical"] = typ_metrics["MdAE"]
    out["MAE_full"] = full_metrics["MAE"]
    out["RMSE_full"] = full_metrics["RMSE"]
    out["MdAE_full"] = full_metrics["MdAE"]
    return out

def build_greedy_forward_specs(ranked_models, pred_fit_df, y_fit):
    specs = []
    search_modes = [
        ("aggregate", "greedy_mean", {"agg_method": "mean"}),
        ("weighted", "greedy_inverse", {"weight_scheme": "inverse_mae"}),
        ("weighted", "greedy_nnls", {"weight_scheme": "nnls"}),
        ("weighted", "greedy_simplex", {"weight_scheme": "simplex_mae"}),
    ]

    for family, prefix, extra in search_modes:
        current = []
        remaining = ranked_models.copy()

        while remaining:
            best = None
            for m in remaining:
                cand = current + [m]
                fit_sub = pred_fit_df[cand].copy()

                if len(cand) == 1:
                    pred = clip_pred(fit_sub.iloc[:, 0].values)
                elif family == "aggregate":
                    pred = aggregate_predictions(fit_sub.values, extra["agg_method"], extra.get("trim_frac", 0.1))
                elif family == "weighted":
                    fitted = fit_weighted_blender(fit_sub, y_fit, extra["weight_scheme"], **extra.get("weight_kwargs", {}))
                    pred = predict_weighted_blender(fitted, fit_sub)
                else:
                    raise KeyError(family)

                score = mae_metric(y_fit, pred)
                if best is None or score < best["score"]:
                    best = {"models": cand.copy(), "score": score}

            current = best["models"]
            remaining = [m for m in remaining if m not in current]

            if len(current) >= 2:
                spec = {"family": family, "name": f"{prefix}_step{len(current)}", "models": current.copy()}
                spec.update(extra)
                specs.append(spec)

    return specs

In [ ]:
# ------------------------------------------------------------
# Generate ensemble specs and score them on the selection split
# ------------------------------------------------------------
base_leaderboard = get_base_leaderboard_from_predictions(val_pred_df, y_blend_val)
display(base_leaderboard.head(20))
save_df(base_leaderboard, "rebuilt_base_leaderboard.csv")

blend_cut = int(len(y_blend_val) * BLEND_FIT_FRAC)
if blend_cut <= 0 or blend_cut >= len(y_blend_val):
    raise ValueError("Некорректный BLEND_FIT_FRAC.")

pred_fit_df = val_pred_df.iloc[:blend_cut].copy()
pred_sel_df = val_pred_df.iloc[blend_cut:].copy()
y_fit = y_blend_val[:blend_cut].copy()
y_sel = y_blend_val[blend_cut:].copy()

ranked_models, fit_rank_df = top_models_by_fit_mae(pred_fit_df, y_fit)
print("Top base learners by blend-fit MAE:")
display(fit_rank_df.head(20))
save_df(fit_rank_df, "blend_fit_model_ranking.csv")

specs = []

# singles
for m in ranked_models:
    specs.append({"family": "single", "name": "single", "models": [m]})

# all-model global aggregations
all_models = ranked_models.copy()
specs.extend([
    {"family": "aggregate", "name": "all_mean", "models": all_models, "agg_method": "mean"},
    {"family": "aggregate", "name": "all_median", "models": all_models, "agg_method": "median"},
    {"family": "aggregate", "name": "all_trim10", "models": all_models, "agg_method": "trimmed_mean", "trim_frac": 0.10},
    {"family": "aggregate", "name": "all_trim20", "models": all_models, "agg_method": "trimmed_mean", "trim_frac": 0.20},
    {"family": "weighted", "name": "all_inverse_mae", "models": all_models, "weight_scheme": "inverse_mae"},
    {"family": "weighted", "name": "all_inverse_mae_sq", "models": all_models, "weight_scheme": "inverse_mae_sq"},
    {"family": "weighted", "name": "all_softmax_t1", "models": all_models, "weight_scheme": "softmax_mae", "weight_kwargs": {"temperature": 1.0}},
    {"family": "weighted", "name": "all_nnls", "models": all_models, "weight_scheme": "nnls"},
    {"family": "weighted", "name": "all_simplex_mae", "models": all_models, "weight_scheme": "simplex_mae"},
])

if RUN_TOPK_PREFIX_ENSEMBLES:
    for k in range(2, len(ranked_models) + 1):
        topk = ranked_models[:k]
        specs.append({"family": "aggregate", "name": f"top{k}_mean", "models": topk, "agg_method": "mean"})
        specs.append({"family": "aggregate", "name": f"top{k}_median", "models": topk, "agg_method": "median"})
        if k >= 5:
            specs.append({"family": "aggregate", "name": f"top{k}_trim10", "models": topk, "agg_method": "trimmed_mean", "trim_frac": 0.10})
        specs.append({"family": "weighted", "name": f"top{k}_inverse_mae", "models": topk, "weight_scheme": "inverse_mae"})
        specs.append({"family": "weighted", "name": f"top{k}_inverse_mae_sq", "models": topk, "weight_scheme": "inverse_mae_sq"})
        for temp in [0.25, 0.5, 1.0, 2.0, 4.0]:
            specs.append({"family": "weighted", "name": f"top{k}_softmax_t{temp}", "models": topk, "weight_scheme": "softmax_mae", "weight_kwargs": {"temperature": temp}})
        for power in [0.5, 1.0, 2.0]:
            specs.append({"family": "weighted", "name": f"top{k}_rankp{power}", "models": topk, "weight_scheme": "rank_power", "weight_kwargs": {"power": power}})
        specs.append({"family": "weighted", "name": f"top{k}_nnls", "models": topk, "weight_scheme": "nnls"})
        specs.append({"family": "weighted", "name": f"top{k}_simplex", "models": topk, "weight_scheme": "simplex_mae"})

if RUN_ALL_PAIRS:
    for a, b in combinations(ranked_models, 2):
        specs.append({"family": "aggregate", "name": "pair_mean", "models": [a, b], "agg_method": "mean"})
        specs.append({"family": "pair_grid", "name": "pair_grid", "models": [a, b]})
        specs.append({"family": "weighted", "name": "pair_nnls", "models": [a, b], "weight_scheme": "nnls"})
        specs.append({"family": "weighted", "name": "pair_simplex", "models": [a, b], "weight_scheme": "simplex_mae"})

if RUN_ALL_TRIPLES:
    for trio in combinations(ranked_models, 3):
        trio = list(trio)
        specs.append({"family": "aggregate", "name": "triple_mean", "models": trio, "agg_method": "mean"})
        specs.append({"family": "aggregate", "name": "triple_median", "models": trio, "agg_method": "median"})
        specs.append({"family": "weighted", "name": "triple_inverse", "models": trio, "weight_scheme": "inverse_mae"})
        specs.append({"family": "weighted", "name": "triple_nnls", "models": trio, "weight_scheme": "nnls"})

if RUN_EXHAUSTIVE_TOPN_SUBSETS:
    topn_models = ranked_models[:min(EXHAUSTIVE_TOP_N, len(ranked_models))]
    for r in range(2, len(topn_models) + 1):
        for subset in combinations(topn_models, r):
            subset = list(subset)
            specs.append({"family": "aggregate", "name": f"subset{r}_mean", "models": subset, "agg_method": "mean"})
            specs.append({"family": "aggregate", "name": f"subset{r}_median", "models": subset, "agg_method": "median"})
            if r >= 5:
                specs.append({"family": "aggregate", "name": f"subset{r}_trim10", "models": subset, "agg_method": "trimmed_mean", "trim_frac": 0.10})
            specs.append({"family": "weighted", "name": f"subset{r}_inverse", "models": subset, "weight_scheme": "inverse_mae"})
            specs.append({"family": "weighted", "name": f"subset{r}_inverse_sq", "models": subset, "weight_scheme": "inverse_mae_sq"})
            specs.append({"family": "weighted", "name": f"subset{r}_softmax", "models": subset, "weight_scheme": "softmax_mae", "weight_kwargs": {"temperature": 1.0}})
            specs.append({"family": "weighted", "name": f"subset{r}_nnls", "models": subset, "weight_scheme": "nnls"})
            specs.append({"family": "weighted", "name": f"subset{r}_simplex", "models": subset, "weight_scheme": "simplex_mae"})

if RUN_GREEDY_SEARCH:
    specs.extend(build_greedy_forward_specs(ranked_models, pred_fit_df, y_fit))

if RUN_STACKERS:
    stackers = ["linear", "positive_linear", "ridge", "lasso", "elastic", "bayes", "huber", "quantile", "gbr", "rf", "etr", "xgb"]
    for stacker_name in stackers:
        specs.append({"family": "stacker", "name": f"all_{stacker_name}", "models": all_models, "stacker_name": stacker_name})
        for k in range(2, len(ranked_models) + 1):
            topk = ranked_models[:k]
            specs.append({"family": "stacker", "name": f"top{k}_{stacker_name}", "models": topk, "stacker_name": stacker_name})

# de-duplicate exact duplicate specs
spec_seen = set()
unique_specs = []
for spec in specs:
    key = json.dumps(
        {
            "family": spec["family"],
            "name": spec["name"],
            "models": spec["models"],
            "agg_method": spec.get("agg_method"),
            "trim_frac": spec.get("trim_frac"),
            "weight_scheme": spec.get("weight_scheme"),
            "weight_kwargs": spec.get("weight_kwargs"),
            "stacker_name": spec.get("stacker_name"),
        },
        sort_keys=True,
        ensure_ascii=False,
    )
    if key not in spec_seen:
        unique_specs.append(spec)
        spec_seen.add(key)

specs = unique_specs
print(f"Total ensemble specs: {len(specs)}")

search_rows = []
for idx, spec in enumerate(specs, start=1):
    if idx % 200 == 0:
        print(f"Selection scoring {idx}/{len(specs)} ...")
    try:
        row = evaluate_spec_on_selection(spec, pred_fit_df, y_fit, pred_sel_df, y_sel)
        search_rows.append(row)
    except Exception as e:
        search_rows.append({
            "tag": spec_tag(spec),
            "family": spec["family"],
            "name": spec["name"],
            "n_models": len(spec["models"]),
            "models": spec["models"],
            "selection_MAE": np.inf,
            "selection_RMSE": np.inf,
            "selection_MdAE": np.inf,
            "error": repr(e),
            "spec": spec,
        })

finite_search_rows = [r for r in search_rows if np.isfinite(r["selection_MAE"])]
finite_search_rows = sorted(
    finite_search_rows,
    key=lambda r: (r["selection_MAE"], r["selection_RMSE"], r["selection_MdAE"], r["tag"])
)

search_df = pd.DataFrame([
    {
        "tag": r["tag"],
        "family": r["family"],
        "name": r["name"],
        "n_models": r["n_models"],
        "models": json.dumps(r["models"], ensure_ascii=False),
        "selection_MAE": r["selection_MAE"],
        "selection_RMSE": r["selection_RMSE"],
        "selection_MdAE": r["selection_MdAE"],
        "error": r.get("error", ""),
        "weights": json.dumps(r.get("weights", []), ensure_ascii=False),
        "weight_first": r.get("weight_first", np.nan),
    }
    for r in finite_search_rows
]).sort_values(["selection_MAE", "selection_RMSE", "selection_MdAE", "tag"], kind="stable").reset_index(drop=True)

display(search_df.head(30))
save_df(search_df, "ensemble_search_leaderboard.csv")

top_refit_rows = finite_search_rows[:min(REFIT_TOP_ENSEMBLES, len(finite_search_rows))]
print(f"Top rows selected for holdout refit: {len(top_refit_rows)}")

In [ ]:
# ------------------------------------------------------------
# Final holdout refit and summary
# ------------------------------------------------------------
final_rows = []
for idx, row in enumerate(top_refit_rows, start=1):
    if idx % 25 == 0:
        print(f"Final holdout refit {idx}/{len(top_refit_rows)} ...")
    spec = row["spec"]
    out = refit_and_eval_spec(
        spec,
        pred_val_df=val_pred_df,
        y_val=y_blend_val,
        pred_test_typ_df=test_typ_fullfit_pred_df,
        y_test_typ=y_test_typ,
        pred_test_full_df=test_full_fullfit_pred_df,
        y_test_full=y_test_full,
    )
    out["selection_MAE"] = row["selection_MAE"]
    out["selection_RMSE"] = row["selection_RMSE"]
    out["selection_MdAE"] = row["selection_MdAE"]
    final_rows.append(out)

final_rows = sorted(
    final_rows,
    key=lambda r: (r["selection_MAE"], r["MAE_typical"], r["MAE_full"], r["tag"])
)

final_df = pd.DataFrame([
    {
        "tag": r["tag"],
        "family": r["family"],
        "name": r["name"],
        "n_models": r["n_models"],
        "models": json.dumps(r["models"], ensure_ascii=False),
        "selection_MAE": r["selection_MAE"],
        "selection_RMSE": r["selection_RMSE"],
        "selection_MdAE": r["selection_MdAE"],
        "MAE_typical": r["MAE_typical"],
        "RMSE_typical": r["RMSE_typical"],
        "MdAE_typical": r["MdAE_typical"],
        "MAE_full": r["MAE_full"],
        "RMSE_full": r["RMSE_full"],
        "MdAE_full": r["MdAE_full"],
        "weights": json.dumps(r.get("weights", []), ensure_ascii=False),
        "weights_like": json.dumps(r.get("weights_like", []), ensure_ascii=False),
        "weight_first": r.get("weight_first", np.nan),
    }
    for r in final_rows
]).sort_values(["selection_MAE", "MAE_typical", "MAE_full", "tag"], kind="stable").reset_index(drop=True)

display(final_df.head(30))
save_df(final_df, "ensemble_holdout_leaderboard.csv")

best_single_base_id = base_leaderboard.iloc[0]["base_id"]
single_typ = score_predictions(y_test_typ, test_typ_fullfit_pred_df[best_single_base_id].values)
single_full = score_predictions(y_test_full, test_full_fullfit_pred_df[best_single_base_id].values)

best_ensemble_row = final_rows[0]

comparison_payload = {
    "selection_metric": "blend_select_MAE",
    "blend_train_frac": BLEND_TRAIN_FRAC,
    "dl_inner_val_frac": DL_INNER_VAL_FRAC,
    "blend_fit_frac": BLEND_FIT_FRAC,
    "n_found_providers": int(len(PROVIDER_REGISTRY)),
    "found_providers": list(PROVIDER_REGISTRY.keys()),
    "provider_selection_rule": "for each (provider, scheme_family, model) choose the best low-level experiment by cv_MAE_internal",
    "model_selection_rule": "for each DL architecture choose the best candidate across provider x scheme_family (up to 12 candidates)",
    "n_candidate_rows": int(len(all_provider_rows)),
    "n_provider_family_model_best_rows": int(len(provider_family_model_best_df)),
    "n_base_learners": int(len(available_base_ids)),
    "available_base_learners": available_base_ids,
    "best_single_base_by_val": best_single_base_id,
    "best_single_base_metrics": {
        "MAE_typical": single_typ["MAE"],
        "RMSE_typical": single_typ["RMSE"],
        "MdAE_typical": single_typ["MdAE"],
        "MAE_full": single_full["MAE"],
        "RMSE_full": single_full["RMSE"],
        "MdAE_full": single_full["MdAE"],
    },
    "best_ensemble": {
        "tag": best_ensemble_row["tag"],
        "family": best_ensemble_row["family"],
        "name": best_ensemble_row["name"],
        "models": best_ensemble_row["models"],
        "selection_MAE": best_ensemble_row["selection_MAE"],
        "MAE_typical": best_ensemble_row["MAE_typical"],
        "RMSE_typical": best_ensemble_row["RMSE_typical"],
        "MdAE_typical": best_ensemble_row["MdAE_typical"],
        "MAE_full": best_ensemble_row["MAE_full"],
        "RMSE_full": best_ensemble_row["RMSE_full"],
        "MdAE_full": best_ensemble_row["MdAE_full"],
        "weights": best_ensemble_row.get("weights", []),
        "weights_like": best_ensemble_row.get("weights_like", []),
        "weight_first": best_ensemble_row.get("weight_first", None),
    },
}

display(pd.DataFrame([
    {"item": "best_single_base_by_val", "value": best_single_base_id},
    {"item": "best_ensemble_tag", "value": best_ensemble_row["tag"]},
    {"item": "best_ensemble_selection_MAE", "value": best_ensemble_row["selection_MAE"]},
    {"item": "best_ensemble_MAE_typical", "value": best_ensemble_row["MAE_typical"]},
    {"item": "best_ensemble_MAE_full", "value": best_ensemble_row["MAE_full"]},
]))

save_json(comparison_payload, "best_ensemble_summary.json")

best_models = best_ensemble_row["models"]
best_weights = best_ensemble_row.get("weights") or best_ensemble_row.get("weights_like") or []
if len(best_weights) == len(best_models) and len(best_weights) > 0:
    wdf = pd.DataFrame({"base_id": best_models, "weight": best_weights})
    wdf = wdf.merge(
        base_pool_df[["base_id", "provider", "scheme_family", "experiment", "model"]],
        on="base_id",
        how="left",
    )
    save_df(wdf, "best_ensemble_weights.csv")

run_manifest = {
    "run_name": RUN_NAME,
    "artifact_root": str(ARTIFACT_ROOT),
    "run_dir": str(RUN_DIR),
    "data_path": str(DATA_PATH),
    "target_col": target_col,
    "provider_manual_artifact_roots": PROVIDER_MANUAL_ARTIFACT_ROOTS,
    "duration_cap": cap,
    "blend_train_frac": BLEND_TRAIN_FRAC,
    "dl_inner_val_frac": DL_INNER_VAL_FRAC,
    "blend_fit_frac": BLEND_FIT_FRAC,
    "keep_only_positive_delta": KEEP_ONLY_POSITIVE_DELTA,
    "dl_inner_val_frac": DL_INNER_VAL_FRAC,
    "max_base_learners": MAX_BASE_LEARNERS,
    "selected_base_learners": available_base_ids,
    "n_selected_base_learners": int(len(available_base_ids)),
    "n_search_specs": int(len(specs)),
    "n_refit_ensembles": int(len(top_refit_rows)),
}

save_json(run_manifest, "run_manifest.json")
print("Saved ensemble artifacts to:", RUN_DIR.resolve())

In [ ]:
# ------------------------------------------------------------
# Графики
# ------------------------------------------------------------
top_plot = final_df.head(15).copy()
plt.figure(figsize=(12, 6))
plt.barh(top_plot["tag"][::-1], top_plot["MAE_typical"][::-1])
plt.xlabel("MAE_typical")
plt.ylabel("Ensemble")
plt.title("Top-15 provider-fusion LLM+DL ensembles by selection_MAE -> holdout typical")
plt.tight_layout()
plt.show()

top_base_plot = base_leaderboard.head(15).copy()
plt.figure(figsize=(12, 6))
plt.barh(top_base_plot["base_id"][::-1], top_base_plot["MAE"][::-1])
plt.xlabel("Blend-val MAE")
plt.ylabel("Base learner")
plt.title("Top-15 selected base learners on blend validation")
plt.tight_layout()
plt.show()

top_corr_models = ranked_models[:min(10, len(ranked_models))]
corr = val_pred_df[top_corr_models].corr()

plt.figure(figsize=(9, 7))
plt.imshow(corr.values, aspect="auto", vmin=-1, vmax=1)
plt.colorbar()
plt.xticks(range(len(top_corr_models)), top_corr_models, rotation=90)
plt.yticks(range(len(top_corr_models)), top_corr_models)
plt.title("Correlation of blend-validation predictions")
plt.tight_layout()
plt.show()

corr.to_csv(RUN_DIR / "blend_val_prediction_correlation.csv")
corr.to_csv(ARTIFACT_ROOT / "blend_val_prediction_correlation_latest.csv")

near_ties = final_df[final_df["selection_MAE"] <= final_df["selection_MAE"].min() + 0.5].copy()
display(near_ties.head(30))
display(final_df.head(20))

## Что смотреть после запуска

Главные артефакты этого финального эксперимента:
- `selected_best12_per_model.csv` — кто победил в логике `4 providers × 3 scheme families -> best of 12`;
- `selected_base_pool.csv` — итоговый reduced pool base learners;
- `available_base_meta.csv` — какие base learners реально успешно восстановились;
- `ensemble_search_leaderboard.csv` — результаты поиска ансамблей на selection split;
- `ensemble_holdout_leaderboard.csv` — финальный holdout leaderboard;
- `best_ensemble_summary.json` — итоговый summary победителя;
- `best_ensemble_weights.csv` — веса лучшего ансамбля, если они есть в явном виде;
- `provider_feature_path_log.csv` — какие именно feature csv были подхвачены;
- `provider_bundle_manifest.csv` — контроль reconstructed feature spaces.
